# Estudo central — classificação multirrótulo, representações, grafos e estruturas relacionais

**Pergunta central:** relacionar obras em um grafo ajuda a classificá-las melhor do que analisar cada sinopse isoladamente?

Este notebook implementa o estudo como uma pipeline em quatro camadas com contratos explícitos:

```text
DatasetContext
      ↓
ExperimentPlan + execute_experiments(...)
      ↓
ExperimentResult  [contrato padronizado]
      ↓
assemble_study_result(...)
      ↓
StudyResult       [única entrada da apresentação]
      ↓
render_study(result)
```

A camada de apresentação **não conhece MiniLM, TF-IDF, RF, XGBoost, MLP, GNN, folds, seeds ou construção do grafo**. Ela recebe somente um `StudyResult` com tabelas, metadados e especificações de visualização. Assim, novas features, modelos ou datasets podem evoluir sem exigir alterações no renderer, desde que o contrato de saída seja preservado.

A auditoria externa permanece reservada e não participa do screening de desenvolvimento.


## 0 — Protocolo e configuração

As escolhas abaixo definem a rodada atual. Elas pertencem à camada experimental; a camada de apresentação não depende delas.

| Item | Regra |
|---|---|
| Desenvolvimento | 304 obras |
| Auditoria | 102 obras reservadas |
| CV | 5 folds agrupados por franquia |
| Threshold | 0,5 fixo |
| Seeds | 42/43/44 para Linear, MLP e GNN; 42 para RF/XGB |
| Hardware | CPU para todos os modelos |
| Grafo | `k=5`, `self_weight=0.90`, sem corte absoluto |
| Métricas primárias | micro-F1 e macro-F1 |
| LLM generativa | não participa do pipeline de classificação |


In [ ]:
import os, sys, json, hashlib, re, time, shutil, importlib.util, subprocess, warnings, platform
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
from itertools import combinations
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.decomposition import TruncatedSVD, NMF
from sklearn.ensemble import RandomForestClassifier

# ---------------- protocolo ----------------
STUDY_VERSION = 'unified_representation_graph_study_v5'
CV_SPLITS = 5
CV_SEED = 42
NEURAL_SEEDS = (42, 43, 44)
CLASSICAL_SEEDS = (42,)
MODEL_SEEDS = {
    'linear': NEURAL_SEEDS,
    'mlp': NEURAL_SEEDS,
    'rf': CLASSICAL_SEEDS,
    'xgb': CLASSICAL_SEEDS,
    'gnn': NEURAL_SEEDS,
    'gcn': NEURAL_SEEDS,
    'graphsage': NEURAL_SEEDS,
    'gatv2': NEURAL_SEEDS,
    'hetero_term_gnn': NEURAL_SEEDS,
    'hetero_term_gatv2': NEURAL_SEEDS,
}
# RF/XGB continuam registrados, mas ficam fora da execução padrão por enquanto.
MODEL_ENABLED = {
    'linear': True,
    'mlp': True,
    'rf': False,
    'xgb': False,
    'gnn': True,
    'gcn': True,              # experimento focal em MiniLM+word_1
    'graphsage': True,        # experimento focal em MiniLM+word_1
    'gatv2': True,            # experimento focal em MiniLM+word_1
    'hetero_term_gnn': True,  # experimento estrutural especial
    'hetero_term_gatv2': True,
}
MAIN_MODEL_ORDER = ('linear','mlp','rf','xgb','gnn')
MODELS = tuple(m for m in MAIN_MODEL_ORDER if MODEL_ENABLED[m])
MODEL_METADATA = {
    'linear': {'display_name':'Linear', 'family':'linear', 'role':'individual', 'enabled':True, 'execution_scope':'main_matrix'},
    'mlp': {'display_name':'MLP', 'family':'neural', 'role':'individual_baseline', 'enabled':True, 'execution_scope':'main_matrix'},
    'rf': {'display_name':'Random Forest', 'family':'tree', 'role':'individual', 'enabled':False, 'execution_scope':'main_matrix'},
    'xgb': {'display_name':'XGBoost', 'family':'boosting', 'role':'individual', 'enabled':False, 'execution_scope':'main_matrix'},
    'gnn': {'display_name':'Fixed-kNN GNN', 'family':'graph', 'role':'relational', 'enabled':True, 'execution_scope':'main_matrix'},
    'gcn': {'display_name':'GCN', 'family':'graph', 'role':'relational_operator', 'enabled':True, 'execution_scope':'special_experiment'},
    'graphsage': {'display_name':'GraphSAGE', 'family':'graph', 'role':'relational_operator', 'enabled':True, 'execution_scope':'special_experiment'},
    'gatv2': {'display_name':'GATv2', 'family':'graph_attention', 'role':'relational_operator', 'enabled':True, 'execution_scope':'special_experiment'},
    'hetero_term_gnn': {'display_name':'Hetero GNN (obra↔termo)', 'family':'heterogeneous_graph', 'role':'relational_structural', 'enabled':True, 'execution_scope':'special_experiment'},
    'hetero_term_gatv2': {'display_name':'Hetero GATv2 (termo→obra)', 'family':'heterogeneous_graph_attention', 'role':'relational_structural', 'enabled':True, 'execution_scope':'special_experiment'},
}
PRIMARY_METRICS = ('f1_micro','f1_macro')
REPORTED_METRICS = ('f1_micro','f1_macro','precision_micro','recall_micro')
CLASS_METRICS = ('precision','recall','f1','support')
METRIC_LABELS = {
    'f1_micro':'Micro-F1', 'f1_macro':'Macro-F1',
    'precision_micro':'Precisão micro', 'recall_micro':'Recall micro',
    'precision':'Precisão', 'recall':'Recall', 'f1':'F1', 'support':'Support',
}
REFERENCE_POLICY = {
    'name':'harmonic_primary_with_stability_penalty_v1',
    'metrics': PRIMARY_METRICS,
    'stability_weight': 0.10,
    'time_tiebreak_component': 'pipeline_incremental',
}
RELATIONAL_DIAGNOSTICS = ({
    'id':'relational_vs_individual',
    'relational_model':'gnn',
    'baseline_model':'mlp',
    'metric':'f1_macro',
    'neighbor_quality_metric':'mean_jaccard',
    'display_name':'Homofilia × ganho relacional',
},)
RESULT_SCHEMA_VERSION = 'study_result_v1'
EXPERIMENT_SCHEMA_VERSION = 'experiment_result_v1'

EPOCHS = 200
HIDDEN = 64
DROPOUT = 0.2
LR = 1e-3
WEIGHT_DECAY = 1e-4
FIXED_THRESHOLD = 0.5
GRAPH_CFG = {'k': 5, 'self_weight': 0.90, 'min_similarity': None}
HOMOGENEOUS_OPERATOR_CFG = {
    'representation_name':'minilm+word_1',
    'bundle':('minilm','word_1'),
    'models':('gcn','graphsage','gatv2'),
    'k':GRAPH_CFG['k'],
    'inductive_policy':'validation nodes receive messages only from training nodes; no validation→training or validation→validation edges',
}
HETERO_DIFFUSION_CFG = {
    'top_k_values':(30,20,10),
    'attention_relation':'term_to_work_only',
    'question':'Does pruning or learned attention reduce diffuse lexical signal in the work↔term graph?',
}
HETERO_TERM_CFG = {
    'id':'hetero_term_gnn_v1',
    'representation_name':'hetero_minilm+term_nodes',
    'work_source':'minilm',
    'term_source':'word_1',
    'term_weight':'tfidf_train_only',
    'hidden':HIDDEN,
    'dropout':DROPOUT,
    'inductive_policy':'term states are built from training works only; validation works never update shared term nodes',
}
# Registro de braços relacionais futuros. Eles fazem parte do protocolo de pesquisa,
# mas NÃO entram no batch atual até serem explicitamente habilitados.
RELATIONAL_RESEARCH_REGISTRY = {
    'fixed_knn_gnn': {
        'status':'active_baseline',
        'question':'A propagação kNN fixa melhora sobre o baseline individual?',
        'structure':'directed train-only kNN; fixed 0.9 self + 0.1 uniform neighbor mean',
    },
    'gcn_knn': {
        'status':'active_focal',
        'question':'A normalização convolucional de GCN melhora sobre a propagação fixa no mesmo kNN?',
        'structure':'same directed inductive kNN edges; PyG GCNConv',
    },
    'graphsage_knn': {
        'status':'active_focal',
        'question':'Separar self e média dos vizinhos via GraphSAGE melhora no mesmo kNN?',
        'structure':'same directed inductive kNN edges; PyG SAGEConv',
    },
    'gatv2_knn': {
        'status':'active_focal',
        'question':'Aprender quanto confiar em cada vizinho melhora no mesmo kNN?',
        'structure':'same directed inductive kNN edges; PyG GATv2Conv',
    },
    'hetero_term_gnn': {
        'status':'active_baseline',
        'question':'Representar termos como nós explícitos melhora em relação a concatenar o sinal lexical?',
        'structure':'work nodes + term nodes; weighted TF-IDF incidence; MiniLM on work nodes',
    },
    'hetero_term_topk': {
        'status':'active_focal',
        'question':'Reduzir o número de termos por obra melhora a heterogênea, como esperado sob a hipótese de sinal difuso?',
        'structure':'same heterogeneous model with per-work TF-IDF top-k pruning',
    },
    'hetero_term_gatv2': {
        'status':'active_focal',
        'question':'Atenção aprendida termo→obra consegue suprimir relações lexicais pouco úteis?',
        'structure':'term states built from training works only; GATv2 attention only on term→work aggregation',
    },
    'hypergraph_terms': {
        'status':'planned_disabled',
        'question':'Relações de alta ordem termo→múltiplas obras são melhores que arestas obra↔obra?',
        'structure':'works as vertices; shared terms/topics as hyperedges',
    },
    'hypergraph_attention': {
        'status':'planned_disabled',
        'question':'Atenção dentro de hiperarestas melhora a seleção de sinal compartilhado?',
        'structure':'hypergraph convolution with attention; only after the non-attentive hypergraph baseline',
    },
}
DEVICE = torch.device('cpu')
CPU_THREADS = max(1, min(4, os.cpu_count() or 1))
torch.set_num_threads(CPU_THREADS)

TARGET = 'Macrogênero (Gênero unificado)'
CLASSES = [
    'Aventura','Ação','Comédia','Cozy','Drama','Esportes','Fantasia',
    'Ficção Científica','Guerra','Mistério','Musical','Realidade','Romance','Terror'
]

DATA_SHA = '55808c037dd84fec48b9f272d2187e6372590ad0b4f083f0acc831695e7c5907'
SPLIT_SHA = '839157ec5a84b8f6542adda81122602529012815fc73bb9e78ddff8eb6a3ed38'
MODEL_ID = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
MODEL_REVISION = 'e8f8c211226b894fcb81acc59f3b34ba3efd5f42'
EMBED_RULE = 'token_chunks_no_overlap_content_weighted_mean_l2_v1'
REFERENCE_MINILM_RUN = '20261021T002245Z_227139'
SPACY_MODEL_ID = 'pt_core_news_sm'
EMOTION_MODEL_ID = 'MilaNLProc/xlm-emo-t'
ENTITY_HASH_DIM = 64
LEX_SVD_DIM = 128
ACTION_SVD_DIM = 64
NMF_DIM = 16

# ---------------- caminhos ----------------
if os.getenv('GNN_PROJECT_ROOT'):
    ROOT = Path(os.environ['GNN_PROJECT_ROOT'])
else:
    try:
        from google.colab import drive
    except ImportError:
        ROOT = Path.cwd()
    else:
        drive.mount('/content/drive')
        ROOT = Path('/content/drive/MyDrive/Pesquisa GNN Midias')

DATA = ROOT/'01_dados'/'medias_classificadas.csv'
PARTS = ROOT/'01_dados'/'particoes_franquias.csv'
LEGACY_PARTS = ROOT/'03_resultados'/'01_1_qualidade_extracao'/'20261019T230939Z'/'particoes_franquias.csv'
if not PARTS.is_file() and LEGACY_PARTS.is_file():
    PARTS.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(LEGACY_PARTS, PARTS)

CENTRAL = ROOT/'03_resultados'/'00_estudo_central'
CACHE = CENTRAL/'cache'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:6]
RUN_DIR = CENTRAL/'runs'/RUN_ID
STUDY_DIR = RUN_DIR/'estudo_unificado'
# Iteração atual: reutiliza somente condições completas da última rodada conhecida,
# após validar hashes do dataset/split e configurações de treino/grafo. Na rodada
# final, defina GNN_INCREMENTAL_REUSE=0 para forçar execução integral do zero.
INCREMENTAL_REUSE = os.getenv('GNN_INCREMENTAL_REUSE','1') == '1'
INCREMENTAL_REUSE_STRICT = os.getenv('GNN_INCREMENTAL_REUSE_STRICT','1') == '1'
INCREMENTAL_BASE_RUN_ID = os.getenv('GNN_INCREMENTAL_BASE_RUN_ID','20261025T142618Z_d9aa19')
CACHE.mkdir(parents=True, exist_ok=True)
STUDY_DIR.mkdir(parents=True, exist_ok=False)

for p in (DATA, PARTS):
    if not p.is_file():
        raise FileNotFoundError(p)


def _ensure_package(import_name, pip_spec=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_spec or import_name], check=True)


def collect_runtime_info():
    ram_gb = None
    try:
        with open('/proc/meminfo', 'r', encoding='utf-8') as f:
            parts = f.readline().split()
        if len(parts) >= 2:
            ram_gb = float(parts[1]) / 1024 / 1024
    except Exception:
        pass
    return {
        'python': sys.version.split()[0],
        'platform': platform.platform(),
        'processor': platform.processor(),
        'cpu_count_logical': os.cpu_count(),
        'cpu_threads_used': CPU_THREADS,
        'ram_gb': ram_gb,
        'torch': torch.__version__,
        'cuda_available_but_not_used': bool(torch.cuda.is_available()),
        'device_for_study': 'cpu',
    }

RUNTIME_INFO = collect_runtime_info()
print(f'Run: {RUN_ID}')
print(f'Device: CPU | threads: {CPU_THREADS} | saída: {STUDY_DIR}')

## 1 — Contratos entre as camadas

Os objetos abaixo são as fronteiras da arquitetura.

- `DatasetContext`: entrada padronizada do estudo;
- `FeatureContext`: representações prontas para os folds;
- `ExperimentPlan`: quais condições serão executadas;
- `ExperimentResult`: saída padronizada da engine, em formato longo;
- `StudyResult`: objeto final e **única dependência do renderer**.

O renderer não acessa variáveis globais do experimento.


In [ ]:
@dataclass(frozen=True)
class DatasetContext:
    frame: pd.DataFrame
    targets: np.ndarray
    classes: tuple
    folds: tuple
    fold_id: np.ndarray
    partitions: pd.DataFrame
    metadata: dict = field(default_factory=dict)


@dataclass
class FeatureContext:
    fold_packs: list
    catalog: pd.DataFrame
    fixed_meta: dict
    fixed_extraction_timing: pd.DataFrame
    fold_meta: pd.DataFrame
    component_timing: pd.DataFrame


@dataclass(frozen=True)
class ExperimentPlan:
    registry: tuple
    model_ids: tuple
    model_seeds: dict
    model_metadata: dict
    primary_metrics: tuple
    reported_metrics: tuple
    class_metrics: tuple
    metric_labels: dict
    reference_policy: dict
    graph_config: dict
    diagnostics: tuple = ()
    special_experiments: tuple = ()
    registered_models: tuple = ()


@dataclass
class ExperimentResult:
    schema_version: str
    conditions: pd.DataFrame
    metric_observations: pd.DataFrame
    class_metric_observations: pd.DataFrame
    timing_observations: pd.DataFrame
    raw_metrics: pd.DataFrame
    raw_class_metrics: pd.DataFrame
    metadata: dict = field(default_factory=dict)


@dataclass
class StudyResult:
    schema_version: str
    metadata: dict
    tables: dict
    reference: dict
    table_specs: list
    plot_specs: list


def stable_id(prefix, value):
    digest = hashlib.sha1(str(value).encode('utf-8')).hexdigest()[:12]
    return f'{prefix}_{digest}'


## 2 — Entrada de dados

Esta camada entrega um `DatasetContext`. A engine experimental recebe esse objeto; a apresentação não conhece o CSV, hashes, colunas originais ou regras de split.


In [ ]:
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def clean_text(text):
    text = re.sub(r'\[\s*(?:Escrito por MAL Reescrever|Written by MAL Rewrite)\s*\]', ' ', str(text), flags=re.I)
    return re.sub(r'\s+', ' ', text).strip()


def load_development(data_path=DATA, split_path=PARTS):
    if sha256(data_path) != DATA_SHA:
        raise ValueError('Dataset diferente da versão congelada.')
    if sha256(split_path) != SPLIT_SHA:
        raise ValueError('Partições diferentes da versão congelada.')

    raw = pd.read_csv(data_path, sep=';', encoding='utf-8-sig', dtype=str, keep_default_na=False)
    parts = pd.read_csv(split_path, dtype=str)
    assert raw.Id.is_unique and parts.Id.is_unique and set(raw.Id) == set(parts.Id)
    assert set(parts.particao) == {'desenvolvimento', 'auditoria'}

    g_dev = set(parts.loc[parts.particao == 'desenvolvimento', 'grupo_franquia'])
    g_aud = set(parts.loc[parts.particao == 'auditoria', 'grupo_franquia'])
    assert not (g_dev & g_aud)

    dev_ids = set(parts.loc[parts.particao == 'desenvolvimento', 'Id'])
    dev = raw.loc[raw.Id.isin(dev_ids), ['Id','Nome','Sinopse','Origem',TARGET]].copy()
    dev = dev.merge(parts[['Id','grupo_franquia']], on='Id', validate='one_to_one')
    dev = dev.sort_values('Id').reset_index(drop=True)
    dev['texto'] = dev.Sinopse.map(clean_text)
    assert len(dev) == 304 and dev.texto.str.len().gt(0).all()

    labels = [[x.strip() for x in s.split(',') if x.strip()] for s in dev[TARGET]]
    unknown = sorted(set(x for row in labels for x in row) - set(CLASSES))
    assert not unknown, f'Classes não previstas: {unknown}'
    mlb = MultiLabelBinarizer(classes=CLASSES)
    y = mlb.fit_transform(labels).astype(np.float32)
    return dev, y, parts


def make_grouped_folds(groups, n_splits=CV_SPLITS, seed=CV_SEED):
    groups = pd.Series(groups, dtype=str).reset_index(drop=True)
    sizes = groups.value_counts().rename_axis('group').reset_index(name='size')
    rng = np.random.default_rng(seed)
    sizes['tie'] = rng.random(len(sizes))
    sizes = sizes.sort_values(['size','tie'], ascending=[False,True]).reset_index(drop=True)

    fold_sizes = np.zeros(n_splits, dtype=int)
    assignment = {}
    for row in sizes.itertuples(index=False):
        f = int(np.argmin(fold_sizes))
        assignment[row.group] = f
        fold_sizes[f] += int(row.size)

    fold_id = groups.map(assignment).to_numpy()
    folds = []
    for f in range(n_splits):
        va = np.where(fold_id == f)[0]
        tr = np.where(fold_id != f)[0]
        assert not set(groups.iloc[tr]) & set(groups.iloc[va])
        folds.append((tr, va))
    return folds, fold_id


DEV, Y, PARTITIONS = load_development()
FOLDS, FOLD_ID = make_grouped_folds(DEV.grupo_franquia)
assert Y.shape == (len(DEV), len(CLASSES))

DATASET = DatasetContext(
    frame=DEV, targets=Y, classes=tuple(CLASSES), folds=tuple(FOLDS), fold_id=FOLD_ID,
    partitions=PARTITIONS,
    metadata={
        'data_sha256': DATA_SHA, 'split_sha256': SPLIT_SHA,
        'n_development': int(len(DEV)),
        'n_audit_reserved': int((PARTITIONS.particao == 'auditoria').sum()),
        'target_column': TARGET, 'group_column': 'grupo_franquia',
    },
)

fold_rows = []
for fold, (tr, va) in enumerate(DATASET.folds, 1):
    support = DATASET.targets[tr].sum(0)
    fold_rows.append({
        'fold': fold, 'n_treino': len(tr), 'n_validacao': len(va),
        'grupos_validacao': DATASET.frame.iloc[va].grupo_franquia.nunique(),
        'menor_suporte_treino': int(support.min()),
    })
FOLD_SUMMARY = pd.DataFrame(fold_rows)
pd.DataFrame({'Id': DATASET.frame.Id, 'fold': DATASET.fold_id + 1}).to_csv(STUDY_DIR/'folds.csv', index=False)

print(f"DatasetContext: {len(DATASET.frame)} desenvolvimento | {DATASET.metadata['n_audit_reserved']} auditoria reservada | {len(DATASET.classes)} classes")
display(FOLD_SUMMARY)


## 3 — Representações

Esta seção **define** os extratores e transformações, mas não executa o estudo ainda. A preparação pesada acontece dentro da pipeline principal por `prepare_feature_context(DATASET)`.

As fontes atuais são MiniLM, n-gramas de palavras/caracteres, entidades, emoção, tópicos, ações e features linguísticas. Transformações corpus-dependentes são ajustadas apenas no treino do fold.


In [ ]:
def safe_unit_rows(x, eps=1e-10):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError('Bloco de features inválido.')
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    out = x.copy()
    mask = norms[:, 0] >= eps
    out[mask] /= norms[mask]
    return out


def encode_documents(texts, encoder, batch_size=32):
    tok = encoder.tokenizer
    if not tok.is_fast:
        raise ValueError('É necessário tokenizer rápido.')
    limit = int(encoder.max_seq_length)
    counts = np.array([
        len(tok(t, add_special_tokens=False, truncation=False, verbose=False)['input_ids'])
        for t in texts
    ], dtype=np.int64)
    if np.any(counts == 0):
        raise ValueError('Sinopse sem tokens.')

    packed = tok(
        list(texts), add_special_tokens=True, truncation=True, max_length=limit,
        stride=0, return_overflowing_tokens=True, return_special_tokens_mask=True,
        return_attention_mask=True, padding=True, return_tensors='pt'
    )
    owner = np.asarray(packed.pop('overflow_to_sample_mapping'), dtype=np.int64)
    special_mask = packed.pop('special_tokens_mask')
    weights = (packed['attention_mask'] * (1 - special_mask)).sum(1).numpy().astype(np.int64)
    covered = np.bincount(owner, weights=weights, minlength=len(texts)).astype(np.int64)
    assert np.array_equal(covered, counts), 'Token perdido ou duplicado.'

    encoder.eval(); encoder.requires_grad_(False)
    chunks = []
    with torch.inference_mode():
        for start in range(0, len(owner), batch_size):
            features = {
                k: v[start:start+batch_size].to(encoder.device)
                for k, v in packed.items()
                if k in ('input_ids','attention_mask','token_type_ids')
            }
            z = encoder(features)['sentence_embedding']
            chunks.append(F.normalize(z.float(), p=2, dim=1).cpu().numpy())
    chunks = np.concatenate(chunks)
    x = np.zeros((len(texts), chunks.shape[1]), dtype=np.float32)
    np.add.at(x, owner, chunks * weights[:, None])
    x /= counts[:, None]
    return safe_unit_rows(x)


def get_minilm_features(dev):
    cache_path = CACHE/'minilm_embeddings.npz'
    legacy = ROOT/'03_resultados'/'02_mvp_mlp_vs_gnn'/REFERENCE_MINILM_RUN/'embeddings_dev.npz'
    for source in (cache_path, legacy):
        if source.is_file():
            with np.load(source, allow_pickle=False) as z:
                ids = z['ids'].astype(str).tolist(); x = z['X'].astype(np.float32)
            if ids == dev.Id.tolist() and x.shape == (len(dev), 384):
                x = safe_unit_rows(x)
                if source != cache_path:
                    np.savez_compressed(cache_path, ids=dev.Id.to_numpy(dtype=str), X=x)
                return x, {'cache_hit': True, 'source': str(source), 'model': MODEL_ID,
                           'revision': MODEL_REVISION, 'rule': EMBED_RULE}

    _ensure_package('sentence_transformers', 'sentence-transformers>=3.4,<6')
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer(MODEL_ID, revision=MODEL_REVISION, device='cpu', trust_remote_code=False)
    x = encode_documents(dev.texto.tolist(), encoder)
    np.savez_compressed(cache_path, ids=dev.Id.to_numpy(dtype=str), X=x)
    return x, {'cache_hit': False, 'source': 'computed', 'model': MODEL_ID,
               'revision': MODEL_REVISION, 'rule': EMBED_RULE, 'device': 'cpu'}


def _resolve_hf_revision(model_id):
    try:
        _ensure_package('huggingface_hub', 'huggingface_hub>=0.25')
        from huggingface_hub import model_info
        return model_info(model_id).sha
    except Exception:
        return 'main'


def _encode_emotion_chunks(texts, tokenizer, model, batch_size=16):
    if not getattr(tokenizer, 'is_fast', False):
        raise ValueError('Tokenizer rápido necessário para agregação de chunks de emoção.')
    max_len = int(getattr(tokenizer, 'model_max_length', 512))
    if max_len > 4096 or max_len < 16:
        max_len = 512
    packed = tokenizer(
        list(texts), add_special_tokens=True, truncation=True, max_length=max_len,
        stride=0, return_overflowing_tokens=True, return_special_tokens_mask=True,
        return_attention_mask=True, padding=True, return_tensors='pt'
    )
    owner = np.asarray(packed.pop('overflow_to_sample_mapping'), dtype=np.int64)
    special = packed.pop('special_tokens_mask')
    weights = (packed['attention_mask'] * (1-special)).sum(1).numpy().astype(np.float32)
    probs = []
    model.eval()
    with torch.inference_mode():
        for start in range(0, len(owner), batch_size):
            batch = {k: v[start:start+batch_size].to(DEVICE) for k,v in packed.items()
                     if k in ('input_ids','attention_mask','token_type_ids')}
            probs.append(torch.softmax(model(**batch).logits.float(), dim=-1).cpu().numpy())
    probs = np.concatenate(probs).astype(np.float32)
    x = np.zeros((len(texts), probs.shape[1]), dtype=np.float32)
    den = np.zeros(len(texts), dtype=np.float32)
    np.add.at(x, owner, probs * weights[:, None]); np.add.at(den, owner, weights)
    x /= np.maximum(den[:, None], 1e-8)
    return x


def get_emotion_features(dev):
    cache = CACHE/'emocao_xlm_emo_t_v1.npz'
    meta_path = CACHE/'emocao_xlm_emo_t_v1.json'
    if cache.is_file():
        with np.load(cache, allow_pickle=False) as z:
            ids = z['ids'].astype(str).tolist(); x = z['X'].astype(np.float32); labels = z['labels'].astype(str).tolist()
        if ids == dev.Id.tolist() and len(x) == len(dev):
            meta = json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.is_file() else {}
            return x, labels, {**meta, 'cache_hit': True, 'source': str(cache)}

    _ensure_package('transformers', 'transformers>=4.45,<6')
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    revision = _resolve_hf_revision(EMOTION_MODEL_ID)
    tokenizer = AutoTokenizer.from_pretrained(EMOTION_MODEL_ID, revision=revision, use_fast=True, trust_remote_code=False)
    model = AutoModelForSequenceClassification.from_pretrained(EMOTION_MODEL_ID, revision=revision, trust_remote_code=False).to(DEVICE)
    x = _encode_emotion_chunks(dev.texto.tolist(), tokenizer, model)
    labels = [str(model.config.id2label[i]) for i in range(model.config.num_labels)]
    np.savez_compressed(cache, ids=dev.Id.to_numpy(dtype=str), X=x, labels=np.asarray(labels, dtype=str))
    meta = {
        'model': EMOTION_MODEL_ID, 'resolved_revision': revision, 'labels': labels,
        'dimensions': int(x.shape[1]),
        'aggregation': 'nonoverlapping_token_chunks_content_weighted_mean_softmax_v1',
        'domain_note': 'multilingual social-media emotion model used frozen on synopses',
    }
    meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')
    return x, labels, {**meta, 'cache_hit': False, 'source': 'computed', 'device': 'cpu'}


def _entity_key(text):
    text = clean_text(text).lower().strip()
    text = re.sub(r'[^\wÀ-ÿ]+', '_', text, flags=re.UNICODE).strip('_')
    return text or 'vazio'


def _safe_lemma(token):
    lemma = (token.lemma_ or token.text).lower().strip()
    return re.sub(r'[^\wÀ-ÿ]+', '_', lemma, flags=re.UNICODE).strip('_')


def get_spacy_text_views(dev):
    """Uma única etapa spaCy para NER, ações e features linguísticas; reutiliza caches legados quando possível."""
    cache = CACHE/f'spacy_text_views_{SPACY_MODEL_ID}_v2.npz'
    meta_path = CACHE/f'spacy_text_views_{SPACY_MODEL_ID}_v2.json'
    if cache.is_file() and meta_path.is_file():
        with np.load(cache, allow_pickle=False) as z:
            ids = z['ids'].astype(str).tolist()
            x_ent = z['X_entities'].astype(np.float32)
            entity_labels = z['entity_labels'].astype(str).tolist()
            action_docs = z['action_docs'].astype(str).tolist()
            x_ling = z['X_linguistic_raw'].astype(np.float32)
            ling_names = z['linguistic_feature_names'].astype(str).tolist()
        if ids == dev.Id.tolist():
            meta = json.loads(meta_path.read_text(encoding='utf-8'))
            return x_ent, entity_labels, action_docs, x_ling, ling_names, {**meta, 'cache_hit': True, 'source': str(cache)}

    # Compatibilidade: os caches produzidos pela versão anterior contêm exatamente estas representações.
    old_ent = CACHE/f'entidades_{SPACY_MODEL_ID}_{ENTITY_HASH_DIM}d_v1.npz'
    old_ent_meta = CACHE/f'entidades_{SPACY_MODEL_ID}_{ENTITY_HASH_DIM}d_v1.json'
    old_narr = CACHE/f'narrativa_{SPACY_MODEL_ID}_v1.npz'
    old_narr_meta = CACHE/f'narrativa_{SPACY_MODEL_ID}_v1.json'
    if old_ent.is_file() and old_narr.is_file():
        with np.load(old_ent, allow_pickle=False) as ze, np.load(old_narr, allow_pickle=False) as zn:
            ids_e = ze['ids'].astype(str).tolist(); ids_n = zn['ids'].astype(str).tolist()
            if ids_e == dev.Id.tolist() == ids_n:
                x_ent = ze['X'].astype(np.float32)
                entity_labels = ze['labels'].astype(str).tolist()
                action_docs = zn['action_docs'].astype(str).tolist()
                x_ling = zn['X_linguistic_raw'].astype(np.float32)
                ling_names = zn['feature_names'].astype(str).tolist()
                np.savez_compressed(
                    cache, ids=dev.Id.to_numpy(dtype=str), X_entities=x_ent,
                    entity_labels=np.asarray(entity_labels, dtype=str), action_docs=np.asarray(action_docs, dtype=str),
                    X_linguistic_raw=x_ling, linguistic_feature_names=np.asarray(ling_names, dtype=str)
                )
                meta = {
                    'model': SPACY_MODEL_ID, 'data_sha256': DATA_SHA,
                    'rule': 'reused_legacy_entity_and_narrative_caches', 'reused_legacy_caches': True,
                    'entity_dimensions': int(x_ent.shape[1]), 'linguistic_dimensions': int(x_ling.shape[1]),
                }
                meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')
                return x_ent, entity_labels, action_docs, x_ling, ling_names, {**meta, 'cache_hit': True, 'source': 'legacy_caches'}

    _ensure_package('spacy', 'spacy>=3.7,<4')
    import spacy
    try:
        nlp = spacy.load(SPACY_MODEL_ID)
    except OSError:
        subprocess.run([sys.executable, '-m', 'spacy', 'download', SPACY_MODEL_ID], check=True)
        nlp = spacy.load(SPACY_MODEL_ID)

    entity_labels = sorted(nlp.get_pipe('ner').labels)
    label_to_i = {lab:i for i,lab in enumerate(entity_labels)}
    pos_names = ['VERB','AUX','NOUN','PROPN','ADJ','ADV','PRON']
    ling_names = [
        'log_tokens','log_sentences','avg_sentence_len','avg_alpha_token_len','lexical_diversity',
        *[f'prop_{p.lower()}' for p in pos_names],
        'negation_ratio','question_ratio','exclamation_ratio','punct_ratio','verb_object_pairs_per_token',
    ]

    entity_docs, type_props, ent_stats, action_docs, ling_rows = [], [], [], [], []
    texts = dev.texto.tolist()
    for text, doc in zip(texts, nlp.pipe(texts, batch_size=32)):
        ents = list(doc.ents)
        counts = np.zeros(len(entity_labels), dtype=np.float32)
        ent_tokens, surfaces = [], []
        for ent in ents:
            key = _entity_key(ent.text)
            ent_tokens.append(f'{ent.label_}::{key}'); surfaces.append(key)
            if ent.label_ in label_to_i:
                counts[label_to_i[ent.label_]] += 1
        total_ent = len(ents)
        n_word = max(1, len(re.findall(r'\w+', text, flags=re.UNICODE)))
        entity_docs.append(' '.join(ent_tokens))
        type_props.append(counts / max(total_ent, 1))
        ent_stats.append([np.log1p(total_ent), total_ent/n_word, len(set(surfaces))/max(total_ent,1), float(total_ent==0)])

        alpha = [t for t in doc if t.is_alpha]
        n_alpha = max(1, len(alpha)); sentences = list(doc.sents); n_sent = max(1, len(sentences))
        pos_counts = {p:0 for p in pos_names}; lemmas = []
        for t in alpha:
            if t.pos_ in pos_counts: pos_counts[t.pos_] += 1
            lem = _safe_lemma(t)
            if lem: lemmas.append(lem)
        action_tokens, pair_count = [], 0
        for t in alpha:
            if t.pos_ != 'VERB': continue
            verb = _safe_lemma(t)
            if not verb: continue
            action_tokens.append(verb)
            for child in t.children:
                if child.dep_ in {'obj','iobj','obl'} and child.pos_ in {'NOUN','PROPN','PRON'}:
                    obj = _safe_lemma(child)
                    if obj:
                        action_tokens.append(f'{verb}__{obj}'); pair_count += 1
        action_docs.append(' '.join(action_tokens) if action_tokens else 'semacao')
        token_count = max(1, len([t for t in doc if not t.is_space]))
        neg_count = sum(1 for t in doc if t.dep_ == 'neg' or _safe_lemma(t) in {'nao','não','nunca','jamais'})
        punct_count = sum(1 for t in doc if t.is_punct)
        ling_rows.append([
            np.log1p(n_alpha), np.log1p(n_sent), n_alpha/n_sent,
            float(np.mean([len(t.text) for t in alpha])) if alpha else 0.0,
            len(set(lemmas))/max(len(lemmas),1),
            *[pos_counts[p]/n_alpha for p in pos_names],
            neg_count/token_count, text.count('?')/token_count, text.count('!')/token_count,
            punct_count/token_count, pair_count/n_alpha,
        ])

    hasher = HashingVectorizer(
        n_features=ENTITY_HASH_DIM, alternate_sign=False, norm=None,
        tokenizer=str.split, preprocessor=None, token_pattern=None, lowercase=False,
    )
    h = hasher.transform(entity_docs).toarray().astype(np.float32)
    x_ent = safe_unit_rows(np.concatenate([h, np.asarray(type_props,np.float32), np.asarray(ent_stats,np.float32)], axis=1))
    x_ling = np.asarray(ling_rows, dtype=np.float32)
    if not np.isfinite(x_ling).all(): raise ValueError('Features linguísticas inválidas.')

    np.savez_compressed(
        cache, ids=dev.Id.to_numpy(dtype=str), X_entities=x_ent,
        entity_labels=np.asarray(entity_labels,dtype=str), action_docs=np.asarray(action_docs,dtype=str),
        X_linguistic_raw=x_ling, linguistic_feature_names=np.asarray(ling_names,dtype=str)
    )
    meta = {
        'model': SPACY_MODEL_ID, 'spacy_version': spacy.__version__, 'data_sha256': DATA_SHA,
        'entity_rule': 'entity_surface_hash_plus_type_proportions_and_counts_v1',
        'action_rule': 'verb_lemmas_plus_simple_dependency_objects_v1',
        'linguistic_rule': 'length_lexical_pos_negation_punctuation_v1',
        'entity_dimensions': int(x_ent.shape[1]), 'linguistic_dimensions': int(x_ling.shape[1]),
        'reused_legacy_caches': False,
    }
    meta_path.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding='utf-8')
    return x_ent, entity_labels, action_docs, x_ling, ling_names, {**meta, 'cache_hit': False, 'source': 'computed'}


def prepare_fixed_features(dataset):
    dev = dataset.frame
    timing_rows = []
    t0=time.perf_counter(); x_minilm,minilm_meta=get_minilm_features(dev); timing_rows.append({'stage':'minilm','observed_seconds':time.perf_counter()-t0,'cache_hit':minilm_meta.get('cache_hit')})
    t0=time.perf_counter(); x_emotion,emotion_labels,emotion_meta=get_emotion_features(dev); timing_rows.append({'stage':'emotion','observed_seconds':time.perf_counter()-t0,'cache_hit':emotion_meta.get('cache_hit')})
    t0=time.perf_counter(); x_entities,entity_labels,action_docs,x_linguistic_raw,linguistic_names,spacy_meta=get_spacy_text_views(dev); timing_rows.append({'stage':'spacy_shared','observed_seconds':time.perf_counter()-t0,'cache_hit':spacy_meta.get('cache_hit')})
    n=len(dev)
    assert len(x_minilm)==len(x_emotion)==len(x_entities)==len(action_docs)==len(x_linguistic_raw)==n
    return {
        'arrays': {'minilm':x_minilm, 'entities':x_entities, 'emotion':x_emotion},
        'action_docs': action_docs, 'linguistic_raw': x_linguistic_raw,
        'labels': {'emotion':emotion_labels, 'entities':entity_labels, 'linguistic':linguistic_names},
        'meta': {'minilm':minilm_meta, 'emotion':emotion_meta, 'spacy':spacy_meta},
        'timing': pd.DataFrame(timing_rows),
    }


In [ ]:
BASE_TFIDF_CFG = dict(
    lowercase=True, strip_accents='unicode', analyzer='word', ngram_range=(1,2),
    token_pattern=r'(?u)\b\w\w+\b', min_df=2, max_df=.95, max_features=20000,
    sublinear_tf=True, smooth_idf=True, norm='l2', dtype=np.float64,
)
WORD_CFGS = {
    'word_1': {**BASE_TFIDF_CFG, 'ngram_range': (1,1)},
    'word_1_2': {**BASE_TFIDF_CFG, 'ngram_range': (1,2)},
    'word_1_3': {**BASE_TFIDF_CFG, 'ngram_range': (1,3)},
}
CHAR_CFG = dict(
    lowercase=True, strip_accents='unicode', analyzer='char_wb', ngram_range=(3,5),
    min_df=2, max_df=.95, max_features=30000, sublinear_tf=True,
    smooth_idf=True, norm='l2', dtype=np.float64,
)
ACTION_CFG = dict(
    lowercase=True, strip_accents='unicode', analyzer='word', ngram_range=(1,2),
    token_pattern=r'(?u)\b\w\w+\b', min_df=2, max_df=1.0, max_features=8000,
    sublinear_tf=True, smooth_idf=True, norm='l2', dtype=np.float64,
)

SOURCE_COMPONENTS = {
    'minilm': (), 'entities': (), 'emotion': (),
    'word_1': ('word_1_base','word_1_svd'),
    'word_1_2': ('word_1_2_base','word_1_2_svd'),
    'word_1_3': ('word_1_3_base','word_1_3_svd'),
    'char_3_5': ('char_3_5_base','char_3_5_svd'),
    'topics': ('word_1_2_base','topics_nmf'),
    'actions': ('actions_base','actions_svd'),
    'linguistic': ('linguistic_scaler',),
}
SOURCE_ORDER = tuple(SOURCE_COMPONENTS)


def _fit_tfidf_base(texts_tr, texts_va, cfg):
    vec = TfidfVectorizer(**cfg)
    t0 = time.perf_counter(); a_tr = vec.fit_transform(texts_tr); fit_s = time.perf_counter()-t0
    if a_tr.shape[1] < 2: raise RuntimeError('Vocabulário TF-IDF insuficiente.')
    t1 = time.perf_counter(); a_va = vec.transform(texts_va); transform_s = time.perf_counter()-t1
    return vec, a_tr, a_va, fit_s, transform_s


def _fit_svd(a_tr, a_va, target_dim, seed=CV_SEED):
    dim = min(target_dim, a_tr.shape[1]-1, a_tr.shape[0]-1)
    svd = TruncatedSVD(n_components=dim, algorithm='randomized', n_iter=7, n_oversamples=10, random_state=seed)
    t0 = time.perf_counter(); xtr = safe_unit_rows(svd.fit_transform(a_tr)); fit_s = time.perf_counter()-t0
    t1 = time.perf_counter(); xva = safe_unit_rows(svd.transform(a_va)); transform_s = time.perf_counter()-t1
    return xtr, xva, svd, fit_s, transform_s


def build_fold_features(dataset, fixed, tr, va, fold_number):
    texts_tr=dataset.frame.iloc[tr].texto.tolist(); texts_va=dataset.frame.iloc[va].texto.tolist()
    arrays=fixed['arrays']
    train={'minilm':arrays['minilm'][tr], 'entities':arrays['entities'][tr], 'emotion':arrays['emotion'][tr]}
    valid={'minilm':arrays['minilm'][va], 'entities':arrays['entities'][va], 'emotion':arrays['emotion'][va]}
    components = {}; meta = {'fold': fold_number}

    # word n-grams; word_1_2 também alimenta NMF para não ajustar o mesmo TF-IDF duas vezes.
    word_bases = {}
    for name, cfg in WORD_CFGS.items():
        _, a_tr, a_va, fit_b, trans_b = _fit_tfidf_base(texts_tr, texts_va, cfg)
        components[f'{name}_base'] = {'fit_seconds': fit_b, 'transform_seconds': trans_b}
        xtr, xva, svd, fit_s, trans_s = _fit_svd(a_tr, a_va, LEX_SVD_DIM)
        components[f'{name}_svd'] = {'fit_seconds': fit_s, 'transform_seconds': trans_s}
        train[name] = xtr; valid[name] = xva
        meta[f'{name}_vocab'] = int(a_tr.shape[1]); meta[f'{name}_dim'] = int(xtr.shape[1])
        meta[f'{name}_evr'] = float(svd.explained_variance_ratio_.sum())
        if name == 'word_1_2': word_bases[name] = (a_tr, a_va)

    # char n-grams
    _, a_tr, a_va, fit_b, trans_b = _fit_tfidf_base(texts_tr, texts_va, CHAR_CFG)
    components['char_3_5_base'] = {'fit_seconds': fit_b, 'transform_seconds': trans_b}
    xtr, xva, svd, fit_s, trans_s = _fit_svd(a_tr, a_va, LEX_SVD_DIM)
    components['char_3_5_svd'] = {'fit_seconds': fit_s, 'transform_seconds': trans_s}
    train['char_3_5'] = xtr; valid['char_3_5'] = xva
    meta['char_3_5_vocab'] = int(a_tr.shape[1]); meta['char_3_5_dim'] = int(xtr.shape[1]); meta['char_3_5_evr'] = float(svd.explained_variance_ratio_.sum())

    # tópicos no mesmo TF-IDF word(1,2)
    a_tr, a_va = word_bases['word_1_2']
    nmf_dim = min(NMF_DIM, a_tr.shape[1], len(tr))
    nmf = NMF(n_components=nmf_dim, init='nndsvda', solver='cd', beta_loss='frobenius', max_iter=500, random_state=CV_SEED)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        t0 = time.perf_counter(); xtr = safe_unit_rows(nmf.fit_transform(a_tr)); fit_s = time.perf_counter()-t0
        t1 = time.perf_counter(); xva = safe_unit_rows(nmf.transform(a_va)); trans_s = time.perf_counter()-t1
    components['topics_nmf'] = {'fit_seconds': fit_s, 'transform_seconds': trans_s}
    train['topics'] = xtr; valid['topics'] = xva; meta['topics_dim'] = int(xtr.shape[1])

    # ações narrativas
    actions_tr=[fixed['action_docs'][i] for i in tr]; actions_va=[fixed['action_docs'][i] for i in va]
    _, a_tr, a_va, fit_b, trans_b = _fit_tfidf_base(actions_tr, actions_va, ACTION_CFG)
    components['actions_base'] = {'fit_seconds': fit_b, 'transform_seconds': trans_b}
    xtr, xva, svd, fit_s, trans_s = _fit_svd(a_tr, a_va, ACTION_SVD_DIM)
    components['actions_svd'] = {'fit_seconds': fit_s, 'transform_seconds': trans_s}
    train['actions'] = xtr; valid['actions'] = xva
    meta['actions_vocab'] = int(a_tr.shape[1]); meta['actions_dim'] = int(xtr.shape[1]); meta['actions_evr'] = float(svd.explained_variance_ratio_.sum())

    # features linguísticas
    scaler = StandardScaler()
    t0 = time.perf_counter(); xtr=scaler.fit_transform(fixed['linguistic_raw'][tr]).astype(np.float32); fit_s = time.perf_counter()-t0
    t1 = time.perf_counter(); xva=scaler.transform(fixed['linguistic_raw'][va]).astype(np.float32); trans_s = time.perf_counter()-t1
    components['linguistic_scaler'] = {'fit_seconds': fit_s, 'transform_seconds': trans_s}
    train['linguistic'] = safe_unit_rows(xtr); valid['linguistic'] = safe_unit_rows(xva)
    meta['linguistic_dim'] = int(xtr.shape[1])

    for source in SOURCE_ORDER:
        if source not in train or train[source].shape[1] != valid[source].shape[1]:
            raise AssertionError(f'Fonte ausente ou dimensão instável: {source}')
        if not np.isfinite(train[source]).all() or not np.isfinite(valid[source]).all():
            raise ValueError(f'NaN/Inf em {source}')
    return {'train': train, 'valid': valid, 'components': components, 'meta': meta}


def prepare_feature_context(dataset):
    print('Preparando extratores congelados/cacheados...')
    fixed=prepare_fixed_features(dataset)
    fixed['timing'].to_csv(STUDY_DIR/'fixed_extraction_timing.csv',index=False)

    fold_packs=[]; feature_meta=[]; component_rows=[]
    print('Preparando transformações fold-local...')
    for fold,(tr,va) in enumerate(dataset.folds,1):
        pack=build_fold_features(dataset,fixed,tr,va,fold)
        fold_packs.append(pack); feature_meta.append(pack['meta'])
        for component,timing in pack['components'].items():
            component_rows.append({'fold':fold,'component':component,**timing})
        print(f'  fold {fold}/{len(dataset.folds)} pronto')

    feature_meta_df=pd.DataFrame(feature_meta)
    component_timing=pd.DataFrame(component_rows)
    feature_meta_df.to_csv(STUDY_DIR/'feature_meta_by_fold.csv',index=False)
    component_timing.to_csv(STUDY_DIR/'feature_component_timing.csv',index=False)

    catalog=[]
    for source in SOURCE_ORDER:
        dims=sorted({p['train'][source].shape[1] for p in fold_packs})
        catalog.append({
            'source':source, 'dim':dims[0] if len(dims)==1 else str(dims),
            'fit_scope':'frozen/cache' if not SOURCE_COMPONENTS[source] else 'outer-train',
            'components':'+'.join(SOURCE_COMPONENTS[source]) or 'precomputed',
        })
    catalog_df=pd.DataFrame(catalog)
    return FeatureContext(
        fold_packs=fold_packs, catalog=catalog_df, fixed_meta=fixed['meta'],
        fixed_extraction_timing=fixed['timing'], fold_meta=feature_meta_df,
        component_timing=component_timing,
    )


## 4 — Modelos, métricas e grafo

A engine conhece os adapters de modelo, inclusive experimentos estruturais que não pertencem ao produto cartesiano bundle × modelo. O `ExperimentResult` não expõe detalhes de implementação: ele registra somente condição, métricas, tempos e metadados padronizados.


In [ ]:
def build_knn_propagation(x_train, x_query, k=5, self_weight=.90, min_similarity=None):
    xt, xq = safe_unit_rows(x_train), safe_unit_rows(x_query)
    n, m = len(xt), len(xq)
    if n < 2 or k < 1 or not (0 <= self_weight <= 1):
        raise ValueError('Configuração de grafo inválida.')

    # Fontes esparsas podem produzir vetor zero quando não há sinal conhecido
    # no fold. Esses nós não devem ganhar vizinhos arbitrários por empate em 0.
    # Política: vetor zero => self-only; vetor zero de treino nunca é candidato.
    eps = 1e-10
    train_has_signal = np.linalg.norm(xt, axis=1) >= eps
    query_has_signal = np.linalg.norm(xq, axis=1) >= eps
    sim_train = xt @ xt.T
    sim_query = xq @ xt.T
    p = np.eye(n, dtype=np.float32)
    pe = np.zeros((n+m, n+m), dtype=np.float32)
    edges = []

    def selected(row, own=None):
        candidates = np.flatnonzero(train_has_signal)
        if own is not None:
            candidates = candidates[candidates != own]
        if len(candidates) == 0:
            return candidates
        order = candidates[np.argsort(-row[candidates], kind='stable')]
        order = order[:min(k, len(order))]
        if min_similarity is not None:
            order = order[row[order] >= min_similarity]
        return order

    for i in range(n):
        nbr = selected(sim_train[i], own=i) if train_has_signal[i] else np.empty(0, dtype=int)
        p[i,:] = 0
        if len(nbr) == 0:
            p[i,i] = 1.0
        else:
            p[i,i] = self_weight
            p[i,nbr] = (1-self_weight)/len(nbr)
        for rank,j in enumerate(nbr,1):
            edges.append(('treino',i,int(j),rank,float(sim_train[i,j])))

    pe[:n,:n] = p
    for q in range(m):
        nbr = selected(sim_query[q]) if query_has_signal[q] else np.empty(0, dtype=int)
        row = n+q
        if len(nbr) == 0:
            pe[row,row] = 1.0
        else:
            pe[row,row] = self_weight
            pe[row,nbr] = (1-self_weight)/len(nbr)
        for rank,j in enumerate(nbr,1):
            edges.append(('validacao',q,int(j),rank,float(sim_query[q,j])))

    assert np.allclose(p.sum(1),1) and np.allclose(pe.sum(1),1)
    assert not pe[:n,n:].any()
    # Zero-signal nodes are guaranteed to be self-only and are never neighbors.
    zero_train = np.flatnonzero(~train_has_signal)
    if len(zero_train):
        assert np.allclose(p[zero_train, zero_train], 1.0)
        assert not any(int(j) in set(zero_train.tolist()) for _,_,j,_,_ in edges)
    zero_query = np.flatnonzero(~query_has_signal)
    for q in zero_query:
        row=n+int(q)
        assert pe[row,row] == 1.0 and np.count_nonzero(pe[row]) == 1

    return p, pe, pd.DataFrame(edges, columns=['subset','destino_local','origem_treino_local','rank','cosseno'])


class LinearClassifier(nn.Module):
    def __init__(self, in_dim, n_labels):
        super().__init__(); self.fc = nn.Linear(in_dim, n_labels)
    def forward(self, x, p=None): return self.fc(x)


class MLPGraphClassifier(nn.Module):
    def __init__(self, in_dim, n_labels, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden); self.fc2 = nn.Linear(hidden, n_labels); self.dropout = nn.Dropout(dropout)
    @staticmethod
    def _propagate(p, x):
        if p is None: return x
        return torch.sparse.mm(p, x) if p.is_sparse else p @ x
    def forward(self, x, p=None):
        h = self._propagate(p, x)
        h = self.dropout(F.relu(self.fc1(h)))
        h = self._propagate(p, h)
        return self.fc2(h)


def _ensure_torch_geometric():
    _ensure_package('torch_geometric', 'torch-geometric>=2.6,<3')
    import torch_geometric
    RUNTIME_INFO['torch_geometric'] = torch_geometric.__version__


def build_knn_edge_index(x_train, x_query, k=5, min_similarity=None):
    """Mesma seleção kNN da GNN fixa, mas retorna apenas arestas j→i.

    Em validação, nós de validação recebem mensagens somente de treino.
    Self-loops são responsabilidade de cada operador (GCN/GATv2); GraphSAGE
    usa seu root path nativo. Isso preserva a topologia de vizinhança sem
    importar o self_weight=0.9 específico do baseline fixo.
    """
    _, _, edges = build_knn_propagation(
        x_train, x_query, k=k, self_weight=0.0, min_similarity=min_similarity
    )
    n = len(x_train)
    src, dst = [], []
    for row in edges.itertuples(index=False):
        src.append(int(row.origem_treino_local))
        dst.append(int(row.destino_local) if row.subset == 'treino' else n + int(row.destino_local))
    if not src:
        return torch.empty((2,0), dtype=torch.long, device=DEVICE)
    return torch.tensor([src,dst], dtype=torch.long, device=DEVICE)


class HomogeneousPyGClassifier(nn.Module):
    def __init__(self, in_dim, n_labels, operator, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        _ensure_torch_geometric()
        from torch_geometric.nn import GATv2Conv, GCNConv, SAGEConv
        self.operator = str(operator)
        self.dropout = nn.Dropout(dropout)
        if self.operator == 'gcn':
            self.conv1 = GCNConv(in_dim, hidden, add_self_loops=True, normalize=True)
            self.conv2 = GCNConv(hidden, n_labels, add_self_loops=True, normalize=True)
        elif self.operator == 'graphsage':
            self.conv1 = SAGEConv(in_dim, hidden, aggr='mean', root_weight=True)
            self.conv2 = SAGEConv(hidden, n_labels, aggr='mean', root_weight=True)
        elif self.operator == 'gatv2':
            self.conv1 = GATv2Conv(in_dim, hidden, heads=1, concat=False, dropout=dropout, add_self_loops=True)
            self.conv2 = GATv2Conv(hidden, n_labels, heads=1, concat=False, dropout=dropout, add_self_loops=True)
        else:
            raise ValueError(operator)

    def forward(self, x, edge_index):
        h = self.conv1(x, edge_index)
        h = self.dropout(F.relu(h))
        return self.conv2(h, edge_index)


def fit_predict_homogeneous_operator(x_train, y_train, x_eval, model, seed, edge_train, edge_eval):
    torch.manual_seed(seed)
    xt=torch.tensor(x_train,dtype=torch.float32,device=DEVICE)
    yt=torch.tensor(y_train,dtype=torch.float32,device=DEVICE)
    xe=torch.tensor(np.concatenate([x_train,x_eval]),dtype=torch.float32,device=DEVICE)
    net=HomogeneousPyGClassifier(xt.shape[1],yt.shape[1],model).to(DEVICE)
    positives=yt.sum(0)
    if (positives==0).any() or (positives==len(yt)).any():
        raise ValueError('Classe degenerada no treino homogêneo.')
    loss_fn=nn.BCEWithLogitsLoss(pos_weight=(len(yt)-positives)/positives)
    opt=torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    t0=time.perf_counter()
    for _ in range(EPOCHS):
        net.train(); opt.zero_grad(set_to_none=True)
        loss=loss_fn(net(xt,edge_train),yt)
        if not torch.isfinite(loss): raise RuntimeError(f'Loss {model} não finita.')
        loss.backward(); opt.step()
    train_s=time.perf_counter()-t0
    t1=time.perf_counter(); net.eval()
    with torch.inference_mode():
        prob=torch.sigmoid(net(xe,edge_eval)[len(xt):]).cpu().numpy()
    pred_s=time.perf_counter()-t1
    return prob,sum(p.numel() for p in net.parameters()),'parameters',train_s,pred_s


CLASSICAL_N_JOBS = CPU_THREADS
RF_CFG = dict(n_estimators=300, max_depth=None, min_samples_leaf=2, max_features='sqrt', class_weight='balanced_subsample', n_jobs=CLASSICAL_N_JOBS)
XGB_CFG = dict(
    n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=1.0, reg_lambda=1.0, objective='binary:logistic', eval_metric='logloss',
    tree_method='hist', n_jobs=CLASSICAL_N_JOBS, verbosity=0,
)
RUNTIME_INFO['sklearn'] = __import__('sklearn').__version__
try:
    import importlib.metadata as _importlib_metadata
    RUNTIME_INFO['xgboost'] = _importlib_metadata.version('xgboost')
except Exception:
    RUNTIME_INFO['xgboost'] = None


def multilabel_metrics(y, prob, threshold=FIXED_THRESHOLD):
    pred = np.asarray(prob) >= threshold
    return {
        'f1_micro': f1_score(y, pred, average='micro', zero_division=0),
        'f1_macro': f1_score(y, pred, average='macro', zero_division=0),
        'precision_micro': precision_score(y, pred, average='micro', zero_division=0),
        'recall_micro': recall_score(y, pred, average='micro', zero_division=0),
    }


def class_metrics(y, prob, classes, threshold=FIXED_THRESHOLD):
    pred=np.asarray(prob)>=threshold
    p,r,f,s=precision_recall_fscore_support(y,pred,average=None,zero_division=0)
    return pd.DataFrame({'class_name':list(classes),'precision':p,'recall':r,'f1':f,'support':s})


def fit_predict_neural(x_train, y_train, x_eval, model, seed, p_train=None, p_eval=None):
    torch.manual_seed(seed)
    xt = torch.tensor(x_train, dtype=torch.float32, device=DEVICE)
    yt = torch.tensor(y_train, dtype=torch.float32, device=DEVICE)
    xe = torch.tensor(np.concatenate([x_train,x_eval]), dtype=torch.float32, device=DEVICE)
    if model == 'linear': net = LinearClassifier(xt.shape[1], yt.shape[1]).to(DEVICE); pt = pe = None
    elif model in ('mlp','gnn'):
        net = MLPGraphClassifier(xt.shape[1], yt.shape[1]).to(DEVICE)
        pt, pe = (p_train,p_eval) if model == 'gnn' else (None,None)
    else: raise ValueError(model)

    positives = yt.sum(0)
    if (positives == 0).any() or (positives == len(yt)).any(): raise ValueError('Classe degenerada no treino.')
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=(len(yt)-positives)/positives)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    t0 = time.perf_counter()
    for _ in range(EPOCHS):
        net.train(); opt.zero_grad(set_to_none=True)
        loss = loss_fn(net(xt, pt), yt)
        if not torch.isfinite(loss): raise RuntimeError('Loss não finita.')
        loss.backward(); opt.step()
    train_s = time.perf_counter()-t0
    t1 = time.perf_counter(); net.eval()
    with torch.inference_mode(): prob = torch.sigmoid(net(xe, pe)[len(xt):]).cpu().numpy()
    pred_s = time.perf_counter()-t1
    return prob, sum(p.numel() for p in net.parameters()), 'parameters', train_s, pred_s


def _positive_prob_from_rf(model, x):
    plist = model.predict_proba(x); cols = []
    for j,p in enumerate(plist):
        classes = np.asarray(model.classes_[j])
        cols.append(p[:,int(np.where(classes==1)[0][0])] if 1 in classes else np.zeros(len(x)))
    return np.column_stack(cols).astype(np.float32)


def fit_predict_classical(x_train, y_train, x_eval, model, seed):
    if model == 'rf':
        clf = RandomForestClassifier(random_state=seed, **RF_CFG)
        t0=time.perf_counter(); clf.fit(x_train,y_train); train_s=time.perf_counter()-t0
        t1=time.perf_counter(); prob=_positive_prob_from_rf(clf,x_eval); pred_s=time.perf_counter()-t1
        complexity=int(sum(t.tree_.node_count for t in clf.estimators_))
        return prob, complexity, 'tree_nodes', train_s, pred_s
    if model == 'xgb':
        _ensure_package('xgboost', 'xgboost>=2.0,<4')
        import xgboost as xgb
        estimators=[]; t0=time.perf_counter()
        for j in range(y_train.shape[1]):
            yj=y_train[:,j].astype(int); pos=max(1,int(yj.sum())); neg=max(1,len(yj)-pos)
            clf=xgb.XGBClassifier(random_state=seed+j, scale_pos_weight=neg/pos, **XGB_CFG)
            clf.fit(x_train,yj); estimators.append(clf)
        train_s=time.perf_counter()-t0
        t1=time.perf_counter(); prob=np.column_stack([clf.predict_proba(x_eval)[:,1] for clf in estimators]).astype(np.float32); pred_s=time.perf_counter()-t1
        complexity=int(sum(clf.get_booster().num_boosted_rounds() for clf in estimators))
        return prob, complexity, 'boosted_trees', train_s, pred_s
    raise ValueError(model)

class HeteroTermGNN(nn.Module):
    """GNN bipartida obra↔termo para classificação dos nós obra.

    Os termos são nós estruturais. Seus estados são agregados exclusivamente a
    partir das obras de treino; validação apenas consulta esses estados, evitando
    fluxo validação→termo→validação.
    """
    def __init__(self, work_dim, n_labels, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        self.work_encoder=nn.Linear(work_dim,hidden)
        self.work_to_term=nn.Linear(hidden,hidden,bias=False)
        self.self_path=nn.Linear(hidden,hidden,bias=False)
        self.term_to_work=nn.Linear(hidden,hidden,bias=False)
        self.dropout=nn.Dropout(dropout)
        self.out=nn.Linear(hidden,n_labels)

    @staticmethod
    def _term_state(h_train, col_normalized_train):
        return col_normalized_train.T @ h_train

    def _work_logits(self, h_work, row_normalized_incidence, term_state):
        msg=row_normalized_incidence @ self.term_to_work(term_state)
        h=F.relu(self.self_path(h_work)+msg)
        h=self.dropout(h)
        return self.out(h)

    def forward_train(self, x_train, row_train, col_train):
        h_train=F.relu(self.work_encoder(x_train))
        term_state=self._term_state(self.work_to_term(h_train),col_train)
        return self._work_logits(h_train,row_train,term_state)

    def forward_eval(self, x_train, x_eval, row_train, col_train, row_eval):
        h_train=F.relu(self.work_encoder(x_train))
        h_eval=F.relu(self.work_encoder(x_eval))
        term_state=self._term_state(self.work_to_term(h_train),col_train)
        return self._work_logits(h_eval,row_eval,term_state)


def prune_tfidf_topk(a, top_k=None):
    """Mantém os top-k pesos TF-IDF de cada obra; não usa rótulos."""
    if top_k is None:
        return a.copy()
    x=a.tocsr(copy=True)
    k=int(top_k)
    if k < 1:
        raise ValueError('top_k deve ser >= 1 ou None.')
    for i in range(x.shape[0]):
        lo,hi=x.indptr[i],x.indptr[i+1]
        if hi-lo <= k:
            continue
        vals=x.data[lo:hi]
        keep=np.argpartition(vals,-k)[-k:]
        mask=np.ones(hi-lo,dtype=bool); mask[keep]=False
        vals[mask]=0.0
    x.eliminate_zeros()
    return x


def prepare_hetero_term_incidence(a_train, a_eval, top_k=None):
    """Incidências TF-IDF normalizadas, opcionalmente podadas por obra."""
    a_train=prune_tfidf_topk(a_train,top_k)
    a_eval=prune_tfidf_topk(a_eval,top_k)
    train=np.asarray(a_train.toarray(),dtype=np.float32)
    eval_=np.asarray(a_eval.toarray(),dtype=np.float32)
    # obra→termo para construir estado dos termos: apenas obras de treino.
    col_sum=train.sum(axis=0,keepdims=True)
    col_norm=np.divide(train,col_sum,out=np.zeros_like(train),where=col_sum>1e-12)
    # termo→obra: pesos TF-IDF normalizados dentro de cada obra.
    row_sum_tr=train.sum(axis=1,keepdims=True)
    row_sum_ev=eval_.sum(axis=1,keepdims=True)
    row_train=np.divide(train,row_sum_tr,out=np.zeros_like(train),where=row_sum_tr>1e-12)
    row_eval=np.divide(eval_,row_sum_ev,out=np.zeros_like(eval_),where=row_sum_ev>1e-12)
    meta={
        'term_nodes':int(train.shape[1]),
        'incidence_train':int(np.count_nonzero(train)),
        'incidence_eval':int(np.count_nonzero(eval_)),
        'zero_term_train_rows':int(np.sum(row_sum_tr[:,0]<=1e-12)),
        'zero_term_eval_rows':int(np.sum(row_sum_ev[:,0]<=1e-12)),
        'term_top_k':None if top_k is None else int(top_k),
    }
    return row_train,col_norm,row_eval,meta


def fit_predict_hetero_term(x_train, y_train, x_eval, row_train, col_train, row_eval, seed):
    torch.manual_seed(seed)
    xt=torch.tensor(x_train,dtype=torch.float32,device=DEVICE)
    yt=torch.tensor(y_train,dtype=torch.float32,device=DEVICE)
    xe=torch.tensor(x_eval,dtype=torch.float32,device=DEVICE)
    rt=torch.tensor(row_train,dtype=torch.float32,device=DEVICE)
    ct=torch.tensor(col_train,dtype=torch.float32,device=DEVICE)
    re=torch.tensor(row_eval,dtype=torch.float32,device=DEVICE)
    net=HeteroTermGNN(xt.shape[1],yt.shape[1],hidden=HETERO_TERM_CFG['hidden'],dropout=HETERO_TERM_CFG['dropout']).to(DEVICE)
    positives=yt.sum(0)
    if (positives==0).any() or (positives==len(yt)).any():
        raise ValueError('Classe degenerada no treino heterogêneo.')
    loss_fn=nn.BCEWithLogitsLoss(pos_weight=(len(yt)-positives)/positives)
    opt=torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    t0=time.perf_counter()
    for _ in range(EPOCHS):
        net.train(); opt.zero_grad(set_to_none=True)
        loss=loss_fn(net.forward_train(xt,rt,ct),yt)
        if not torch.isfinite(loss): raise RuntimeError('Loss heterogênea não finita.')
        loss.backward(); opt.step()
    train_s=time.perf_counter()-t0
    t1=time.perf_counter(); net.eval()
    with torch.inference_mode():
        prob=torch.sigmoid(net.forward_eval(xt,xe,rt,ct,re)).cpu().numpy()
    pred_s=time.perf_counter()-t1
    return prob,sum(p.numel() for p in net.parameters()),'parameters',train_s,pred_s



def _term_to_work_edges(row_incidence):
    arr=np.asarray(row_incidence,dtype=np.float32)
    work_idx,term_idx=np.nonzero(arr>0)
    if len(work_idx)==0:
        edge_index=torch.empty((2,0),dtype=torch.long,device=DEVICE)
        edge_attr=torch.empty((0,1),dtype=torch.float32,device=DEVICE)
    else:
        edge_index=torch.tensor(np.vstack([term_idx,work_idx]),dtype=torch.long,device=DEVICE)
        edge_attr=torch.tensor(arr[work_idx,term_idx,None],dtype=torch.float32,device=DEVICE)
    return edge_index,edge_attr


class HeteroTermGATv2(nn.Module):
    """Mantém termo-state train-only e troca apenas termo→obra por GATv2.

    Esta variante foi desenhada especificamente para testar a hipótese de sinal
    lexical difuso: a construção dos estados dos termos permanece idêntica ao
    baseline; somente a confiança em cada termo conectado à obra é aprendida.
    """
    def __init__(self, work_dim, n_labels, hidden=HIDDEN, dropout=DROPOUT):
        super().__init__()
        _ensure_torch_geometric()
        from torch_geometric.nn import GATv2Conv
        self.work_encoder=nn.Linear(work_dim,hidden)
        self.work_to_term=nn.Linear(hidden,hidden,bias=False)
        self.self_path=nn.Linear(hidden,hidden,bias=False)
        self.term_attention=GATv2Conv(
            (hidden,hidden),hidden,heads=1,concat=False,dropout=dropout,
            add_self_loops=False,edge_dim=1,
        )
        self.dropout=nn.Dropout(dropout)
        self.out=nn.Linear(hidden,n_labels)

    def _term_state(self,h_train,col_train):
        return col_train.T @ self.work_to_term(h_train)

    def _logits(self,h_work,term_state,edge_index,edge_attr):
        msg=self.term_attention((term_state,h_work),edge_index,edge_attr=edge_attr)
        h=self.dropout(F.relu(self.self_path(h_work)+msg))
        return self.out(h)

    def forward_train(self,x_train,col_train,edge_index,edge_attr):
        h_train=F.relu(self.work_encoder(x_train))
        term_state=self._term_state(h_train,col_train)
        return self._logits(h_train,term_state,edge_index,edge_attr)

    def forward_eval(self,x_train,x_eval,col_train,edge_index_eval,edge_attr_eval):
        h_train=F.relu(self.work_encoder(x_train))
        h_eval=F.relu(self.work_encoder(x_eval))
        term_state=self._term_state(h_train,col_train)
        return self._logits(h_eval,term_state,edge_index_eval,edge_attr_eval)


def fit_predict_hetero_term_gatv2(x_train,y_train,x_eval,row_train,col_train,row_eval,seed):
    torch.manual_seed(seed)
    xt=torch.tensor(x_train,dtype=torch.float32,device=DEVICE)
    yt=torch.tensor(y_train,dtype=torch.float32,device=DEVICE)
    xe=torch.tensor(x_eval,dtype=torch.float32,device=DEVICE)
    ct=torch.tensor(col_train,dtype=torch.float32,device=DEVICE)
    edge_tr,attr_tr=_term_to_work_edges(row_train)
    edge_ev,attr_ev=_term_to_work_edges(row_eval)
    net=HeteroTermGATv2(xt.shape[1],yt.shape[1],hidden=HETERO_TERM_CFG['hidden'],dropout=HETERO_TERM_CFG['dropout']).to(DEVICE)
    positives=yt.sum(0)
    if (positives==0).any() or (positives==len(yt)).any():
        raise ValueError('Classe degenerada no treino heterogêneo GATv2.')
    loss_fn=nn.BCEWithLogitsLoss(pos_weight=(len(yt)-positives)/positives)
    opt=torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    t0=time.perf_counter()
    for _ in range(EPOCHS):
        net.train(); opt.zero_grad(set_to_none=True)
        loss=loss_fn(net.forward_train(xt,ct,edge_tr,attr_tr),yt)
        if not torch.isfinite(loss): raise RuntimeError('Loss heterogênea GATv2 não finita.')
        loss.backward(); opt.step()
    train_s=time.perf_counter()-t0
    t1=time.perf_counter(); net.eval()
    with torch.inference_mode():
        prob=torch.sigmoid(net.forward_eval(xt,xe,ct,edge_ev,attr_ev)).cpu().numpy()
    pred_s=time.perf_counter()-t1
    return prob,sum(p.numel() for p in net.parameters()),'parameters',train_s,pred_s


## 5 — Plano experimental

O plano registra **o que executar**, sem qualquer dependência da camada de visualização. Hoje são 55 bundles e cinco modelos; amanhã esse registro pode mudar sem alterar `render_study`.


In [ ]:
def _dedupe_registry(rows):
    seen={}; out=[]
    for family,bundle in rows:
        bundle=tuple(bundle); key='+'.join(bundle)
        if key not in seen:
            seen[key]=family; out.append({'family':family,'bundle':bundle,'features':key})
    return out

CORE_SOURCES = ('minilm','word_1_2','entities','emotion','topics')
registry_rows=[]
for r in range(1,len(CORE_SOURCES)+1):
    for combo in combinations(CORE_SOURCES,r): registry_rows.append(('core_factorial',combo))

registry_rows += [
    ('lexical',('word_1',)), ('lexical',('word_1_3',)), ('lexical',('char_3_5',)),
    ('lexical',('word_1_2','char_3_5')), ('lexical',('word_1_3','char_3_5')),
    ('lexical',('minilm','word_1')), ('lexical',('minilm','word_1_3')), ('lexical',('minilm','char_3_5')),
    ('lexical',('minilm','word_1_2','char_3_5')), ('lexical',('minilm','word_1_3','char_3_5')),
]
registry_rows += [
    ('narrative',('actions',)), ('narrative',('linguistic',)),
    ('narrative',('actions','emotion')), ('narrative',('actions','topics')),
    ('narrative',('actions','emotion','topics')), ('narrative',('actions','linguistic')),
    ('narrative',('linguistic','emotion')), ('narrative',('linguistic','topics')),
    ('narrative',('minilm','actions')), ('narrative',('minilm','linguistic')),
    ('narrative',('minilm','actions','emotion')), ('narrative',('minilm','actions','topics')),
    ('narrative',('minilm','actions','linguistic')), ('narrative',('minilm','actions','emotion','topics')),
]

BUNDLE_REGISTRY = _dedupe_registry(registry_rows)
for row in BUNDLE_REGISTRY:
    assert all(s in SOURCE_ORDER for s in row['bundle'])
BUNDLE_TABLE=pd.DataFrame([{**r,'bundle':'|'.join(r['bundle'])} for r in BUNDLE_REGISTRY])
BUNDLE_TABLE.to_csv(STUDY_DIR/'bundle_registry.csv',index=False)


SPECIAL_EXPERIMENTS = (
    # 1) Homogênea: mesmo vetor, mesmo kNN, muda somente o operador.
    {
        'id':'homogeneous_gcn_minilm_word1_v1','adapter':'homogeneous_operator','family':'homogeneous_operator',
        'features':'minilm+word_1','bundle':('minilm','word_1'),'model_id':'gcn','fontes':'minilm|word_1','n_fontes':2,
        'enabled':True,'description':'GCNConv no mesmo kNN indutivo usado pelo baseline Fixed-kNN GNN',
    },
    {
        'id':'homogeneous_graphsage_minilm_word1_v1','adapter':'homogeneous_operator','family':'homogeneous_operator',
        'features':'minilm+word_1','bundle':('minilm','word_1'),'model_id':'graphsage','fontes':'minilm|word_1','n_fontes':2,
        'enabled':True,'description':'SAGEConv(mean) no mesmo kNN indutivo usado pelo baseline Fixed-kNN GNN',
    },
    {
        'id':'homogeneous_gatv2_minilm_word1_v1','adapter':'homogeneous_operator','family':'homogeneous_operator',
        'features':'minilm+word_1','bundle':('minilm','word_1'),'model_id':'gatv2','fontes':'minilm|word_1','n_fontes':2,
        'enabled':True,'description':'GATv2Conv no mesmo kNN indutivo; atenção aprende quanto confiar em cada vizinho',
    },

    # 2) Heterogênea: baseline atual e testes da hipótese de sinal lexical difuso.
    {
        'id':'hetero_term_gnn_v1','adapter':'hetero_term','family':'heterogeneous',
        'features':HETERO_TERM_CFG['representation_name'],'model_id':'hetero_term_gnn','fontes':'minilm|word_1_terms','n_fontes':2,
        'top_k_terms':None,'enabled':True,
        'description':'Baseline atual: MiniLM nas obras + todos os nós termo, incidência TF-IDF train-only',
    },
    *tuple({
        'id':f'hetero_term_gnn_top{k}_v1','adapter':'hetero_term','family':'heterogeneous_diffusion',
        'features':f"{HETERO_TERM_CFG['representation_name']}_top{k}",'model_id':'hetero_term_gnn',
        'fontes':'minilm|word_1_terms','n_fontes':2,'top_k_terms':k,'enabled':True,
        'description':f'Mesma hetero, mantendo apenas os top-{k} termos TF-IDF por obra',
    } for k in HETERO_DIFFUSION_CFG['top_k_values']),
    {
        'id':'hetero_term_gatv2_v1','adapter':'hetero_term','family':'heterogeneous_attention',
        'features':HETERO_TERM_CFG['representation_name'],'model_id':'hetero_term_gatv2','fontes':'minilm|word_1_terms','n_fontes':2,
        'top_k_terms':None,'enabled':True,
        'description':'Estados dos termos train-only; GATv2 aprende a agregação termo→obra usando TF-IDF como atributo de aresta',
    },
)


EXPERIMENT_PLAN=ExperimentPlan(
    registry=tuple(BUNDLE_REGISTRY), model_ids=tuple(MODELS),
    model_seeds={k:tuple(v) for k,v in MODEL_SEEDS.items()},
    model_metadata=MODEL_METADATA, primary_metrics=PRIMARY_METRICS,
    reported_metrics=REPORTED_METRICS, class_metrics=CLASS_METRICS,
    metric_labels=METRIC_LABELS, reference_policy=REFERENCE_POLICY,
    graph_config=GRAPH_CFG, diagnostics=RELATIONAL_DIAGNOSTICS,
    special_experiments=tuple(x for x in SPECIAL_EXPERIMENTS if x.get('enabled',True)),
    registered_models=tuple(MODEL_METADATA),
)

main_conditions=len(EXPERIMENT_PLAN.registry)*len(EXPERIMENT_PLAN.model_ids)
special_conditions=len(EXPERIMENT_PLAN.special_experiments)
n_conditions=main_conditions+special_conditions
main_fits=sum(len(DATASET.folds)*len(EXPERIMENT_PLAN.model_seeds[m]) for m in EXPERIMENT_PLAN.model_ids)*len(EXPERIMENT_PLAN.registry)
special_fits=sum(len(DATASET.folds)*len(EXPERIMENT_PLAN.model_seeds[e['model_id']]) for e in EXPERIMENT_PLAN.special_experiments)
n_fits=main_fits+special_fits
print(f'ExperimentPlan: {len(EXPERIMENT_PLAN.registry)} bundles × {len(EXPERIMENT_PLAN.model_ids)} modelos ativos + {special_conditions} experimento(s) estrutural(is) | {n_conditions} condições | {n_fits} fits fold/seed')
print('Modelos ativos na matriz:', ', '.join(EXPERIMENT_PLAN.model_ids))
print('Registrados mas desabilitados:', ', '.join(m for m,v in MODEL_METADATA.items() if not v.get('enabled',True)) or 'nenhum')
display(BUNDLE_TABLE.groupby('family').size().rename('bundles').reset_index())
display(pd.DataFrame(EXPERIMENT_PLAN.special_experiments))


## 6 — Sanity checks

Os sanity checks recebem os objetos da pipeline e não executam automaticamente durante a definição das funções.


O experimento heterogêneo também valida a política indutiva: termos são ajustados somente no treino externo e os nós de validação nunca atualizam estados compartilhados.


In [ ]:
def combine_blocks(block_dict, sources):
    # Vetor zero é um estado válido para fontes esparsas: significa que a obra
    # não possui sinal conhecido naquele fold (ex.: ações removidas por min_df).
    # safe_unit_rows preserva esses vetores como zero; modelos não relacionais
    # podem consumi-los normalmente. A política especial de vizinhança fica no
    # construtor do grafo, que usa self-only para nós sem sinal.
    blocks=[safe_unit_rows(block_dict[s]) for s in sources]
    x=safe_unit_rows(np.concatenate(blocks,axis=1).astype(np.float32))
    if x.ndim!=2 or x.shape[1]==0 or not np.isfinite(x).all():
        raise ValueError(f'Bloco combinado inválido em {sources}')
    return x


def bundle_component_time(pack, bundle):
    names=[]
    for source in bundle: names.extend(SOURCE_COMPONENTS[source])
    unique=list(dict.fromkeys(names))
    fit=sum(float(pack['components'][c]['fit_seconds']) for c in unique)
    transform=sum(float(pack['components'][c]['transform_seconds']) for c in unique)
    return fit,transform,unique

def run_sanity_checks(dataset, feature_context, plan):
    for (tr,va),pack in zip(dataset.folds,feature_context.fold_packs):
        assert not set(dataset.frame.iloc[tr].grupo_franquia) & set(dataset.frame.iloc[va].grupo_franquia)
        for src in SOURCE_ORDER:
            assert pack['train'][src].shape[1]==pack['valid'][src].shape[1]

    tr,va=dataset.folds[0]
    xtr=combine_blocks(feature_context.fold_packs[0]['train'],('minilm',))
    xva=combine_blocks(feature_context.fold_packs[0]['valid'],('minilm',))
    p,pe,_=build_knn_propagation(xtr,xva,**plan.graph_config)
    assert p.shape==(len(tr),len(tr)) and pe.shape==(len(tr)+len(va),len(tr)+len(va))
    assert not pe[:len(tr),len(tr):].any()

    # Regressão: fontes esparsas podem ter linhas zero. Para modelos comuns isso
    # é ausência de sinal; para o grafo elas devem ficar self-only e nunca virar
    # vizinhas de outros nós.
    ztr=np.array([[1.,0.],[0.,0.],[0.,1.]],dtype=np.float32)
    zva=np.array([[0.,0.],[1.,1.]],dtype=np.float32)
    zp,zpe,zedges=build_knn_propagation(ztr,zva,k=2,self_weight=.9,min_similarity=None)
    assert np.count_nonzero(zp[1])==1 and zp[1,1]==1.0
    assert np.count_nonzero(zpe[len(ztr)])==1 and zpe[len(ztr),len(ztr)]==1.0
    assert 1 not in set(zedges.origem_treino_local.astype(int))

    torch.manual_seed(123)
    a=MLPGraphClassifier(8,4,hidden=5,dropout=0.0); b=MLPGraphClassifier(8,4,hidden=5,dropout=0.0)
    b.load_state_dict(a.state_dict()); z=torch.randn(7,8); ident=torch.eye(7).to_sparse()
    assert torch.allclose(a(z,None),b(z,ident),atol=1e-7)
    hetero_meta=sanity_check_hetero_term(dataset)
    assert hetero_meta['term_nodes']>0
    return True



def sanity_check_hetero_term(dataset):
    tr,va=dataset.folds[0]
    texts_tr=dataset.frame.iloc[tr].texto.tolist(); texts_va=dataset.frame.iloc[va].texto.tolist()
    _,a_tr,a_va,_,_=_fit_tfidf_base(texts_tr,texts_va,WORD_CFGS['word_1'])
    rt,ct,re,meta=prepare_hetero_term_incidence(a_tr,a_va)
    assert rt.shape==(len(tr),a_tr.shape[1])
    assert ct.shape==rt.shape and re.shape==(len(va),a_tr.shape[1])
    # colunas com incidência têm soma 1 na direção obra→termo agregadora.
    nz=a_tr.toarray().sum(axis=0)>1e-12
    if np.any(nz): assert np.allclose(ct[:,nz].sum(axis=0),1.0,atol=1e-5)
    assert meta['term_nodes']==a_tr.shape[1]
    return meta


## 7 — Engine experimental → `ExperimentResult`

A engine pode mudar internamente, inclusive adicionando famílias estruturais especiais, mas sua saída pública é estável:

- `conditions`: catálogo das condições;
- `metric_observations`: métricas globais em formato longo;
- `class_metric_observations`: métricas por classe em formato longo;
- `timing_observations`: tempos por componente em formato longo.

Essas tabelas formam o contrato entre experimentação e análise.


In [ ]:
def _expected_rows_for_model(dataset, plan, model):
    return len(dataset.folds)*len(plan.model_seeds[model])


def _completed_conditions(df, dataset, plan):
    if df is None or df.empty:
        return set()
    done=set()
    for (features,model),g in df.groupby(['features','modelo']):
        if len(g)==_expected_rows_for_model(dataset,plan,model):
            done.add((features,model))
    return done


def _condition_table_from_raw(raw_metrics, plan):
    cols=['family','features','fontes','n_fontes','input_dim','modelo']
    x=raw_metrics[cols].drop_duplicates().copy()
    x['representation_id']=x.features.map(lambda v: stable_id('repr',v))
    x['condition_id']=[stable_id('cond',f'{r.features}|{r.modelo}') for r in x.itertuples(index=False)]
    x['representation_name']=x.features.astype(str)
    x['model_id']=x.modelo.astype(str)
    x['model_name']=x.model_id.map(lambda m: plan.model_metadata.get(m,{}).get('display_name',m))
    x['model_family']=x.model_id.map(lambda m: plan.model_metadata.get(m,{}).get('family','unknown'))
    x['model_role']=x.model_id.map(lambda m: plan.model_metadata.get(m,{}).get('role','unknown'))
    x['execution_scope']=x.model_id.map(lambda m: plan.model_metadata.get(m,{}).get('execution_scope','main_matrix'))
    x['model_enabled']=x.model_id.map(lambda m: bool(plan.model_metadata.get(m,{}).get('enabled',True)))
    return x[['condition_id','representation_id','representation_name','model_id','model_name','model_family','model_role','execution_scope','model_enabled','family','fontes','n_fontes','input_dim']].sort_values(['representation_name','model_id']).reset_index(drop=True)


def standardize_experiment_result(raw_metrics, raw_class_metrics, plan):
    conditions=_condition_table_from_raw(raw_metrics,plan)
    key_map=conditions.set_index(['representation_name','model_id']).condition_id.to_dict()

    m=raw_metrics.copy()
    m['condition_id']=[key_map[(str(f),str(mod))] for f,mod in zip(m.features,m.modelo)]
    metric_cols=[c for c in plan.reported_metrics if c in m.columns]
    metric_obs=m.melt(id_vars=['condition_id','fold','seed'],value_vars=metric_cols,var_name='metric',value_name='value')

    cm=raw_class_metrics.copy()
    cm['condition_id']=[key_map[(str(f),str(mod))] for f,mod in zip(cm.features,cm.modelo)]
    class_cols=[c for c in plan.class_metrics if c in cm.columns]
    class_obs=cm.melt(id_vars=['condition_id','fold','seed','class_name'],value_vars=class_cols,var_name='metric',value_name='value')

    time_map={
        'feature_fit_seconds_incremental':'feature_fit',
        'feature_transform_seconds_incremental':'feature_transform',
        'feature_fusion_seconds':'feature_fusion',
        'graph_build_seconds':'graph_build',
        'train_seconds':'train', 'predict_seconds':'predict',
        'model_seconds':'model', 'pipeline_seconds_incremental':'pipeline_incremental',
        'condition_wall_seconds_observed':'condition_wall',
    }
    present={k:v for k,v in time_map.items() if k in m.columns}
    timing=m.melt(id_vars=['condition_id','fold','seed'],value_vars=list(present),var_name='_raw_component',value_name='seconds')
    timing['component']=timing._raw_component.map(present)
    timing=timing.drop(columns='_raw_component')

    return ExperimentResult(
        schema_version=EXPERIMENT_SCHEMA_VERSION,
        conditions=conditions,
        metric_observations=metric_obs,
        class_metric_observations=class_obs,
        timing_observations=timing,
        raw_metrics=raw_metrics,
        raw_class_metrics=raw_class_metrics,
        metadata={
            'primary_metrics':list(plan.primary_metrics),
            'metric_labels':dict(plan.metric_labels),
            'reference_policy':dict(plan.reference_policy),
            'model_metadata':plan.model_metadata,
        },
    )


def persist_experiment_result(result, directory):
    directory=Path(directory); directory.mkdir(parents=True,exist_ok=True)
    result.conditions.to_csv(directory/'conditions.csv',index=False)
    result.metric_observations.to_csv(directory/'metric_observations.csv',index=False)
    result.class_metric_observations.to_csv(directory/'class_metric_observations.csv',index=False)
    result.timing_observations.to_csv(directory/'timing_observations.csv',index=False)


def _current_training_signature():
    return {
        'epochs':EPOCHS,'hidden':HIDDEN,'dropout':DROPOUT,'lr':LR,
        'weight_decay':WEIGHT_DECAY,'threshold':FIXED_THRESHOLD,
        'loss':'BCEWithLogitsLoss_pos_weight_train_only',
    }


def _allowed_condition_pairs(plan):
    allowed={(str(e['features']),str(m)) for e in plan.registry for m in plan.model_ids}
    allowed.update((str(e['features']),str(e['model_id'])) for e in plan.special_experiments)
    return allowed


def load_incremental_base(dataset, plan):
    """Carrega resultados completos da rodada-base compatível.

    Em modo incremental estrito, qualquer falha de localização/compatibilidade
    interrompe a execução em vez de rerodar silenciosamente a matriz antiga.
    """
    if not INCREMENTAL_REUSE or not INCREMENTAL_BASE_RUN_ID:
        print('Reuso incremental desabilitado; execução integral solicitada.')
        return pd.DataFrame(),pd.DataFrame()

    base=CENTRAL/'runs'/INCREMENTAL_BASE_RUN_ID/'estudo_unificado'
    manifest_path=base/'manifesto.json'
    metrics_path=base/'_internal_metrics_partial.csv'
    class_path=base/'_internal_class_metrics_partial.csv'
    print(f'Base incremental solicitada: {base}')

    missing=[str(p) for p in (manifest_path,metrics_path,class_path) if not p.is_file()]
    if missing:
        msg='Checkpoint incremental incompleto/não encontrado: ' + '; '.join(missing)
        if INCREMENTAL_REUSE_STRICT:
            raise FileNotFoundError(msg)
        print(msg+'; execução seguirá sem reuso.')
        return pd.DataFrame(),pd.DataFrame()

    try:
        manifest=json.loads(manifest_path.read_text(encoding='utf-8'))
        ds=manifest.get('dataset',{})
        checks={
            'dataset_hash':ds.get('data_sha256')==DATA_SHA,
            'split_hash':ds.get('split_sha256')==SPLIT_SHA,
            'audit_unused':manifest.get('audit_used') is False,
            'cv_splits':manifest.get('cv',{}).get('outer_splits')==len(dataset.folds),
            'fold_seed':manifest.get('cv',{}).get('fold_seed')==CV_SEED,
            'training':manifest.get('training')==_current_training_signature(),
            'graph':manifest.get('graph')==plan.graph_config,
        }
        print('Compatibilidade incremental:', ', '.join(f'{k}={"OK" if v else "FAIL"}' for k,v in checks.items()))
        if not all(checks.values()):
            bad=[k for k,v in checks.items() if not v]
            msg='Checkpoint incremental rejeitado por incompatibilidade: '+', '.join(bad)
            if INCREMENTAL_REUSE_STRICT:
                raise RuntimeError(msg)
            print(msg+'; execução seguirá sem reuso.')
            return pd.DataFrame(),pd.DataFrame()

        raw=pd.read_csv(metrics_path)
        cls=pd.read_csv(class_path)
        base_pairs=raw[['features','modelo']].drop_duplicates().shape[0]
        base_counts=raw.groupby(['features','modelo']).size()
        malformed=base_counts[base_counts != len(dataset.folds)*3]
        if len(malformed):
            print(f'Aviso: {len(malformed)} pares da base não têm 15 linhas; serão ignorados se incompletos.')

        allowed=_allowed_condition_pairs(plan)
        mask=[(str(f),str(m)) in allowed for f,m in zip(raw.features,raw.modelo)]
        raw=raw.loc[mask].copy()
        done=_completed_conditions(raw,dataset,plan)
        raw=raw[[ (str(f),str(m)) in done for f,m in zip(raw.features,raw.modelo)]].copy()
        if not cls.empty:
            cls=cls[[ (str(f),str(m)) in done for f,m in zip(cls.features,cls.modelo)]].copy()

        if not done:
            msg=f'Nenhuma condição completa reutilizável encontrada na base ({base_pairs} pares brutos).'
            if INCREMENTAL_REUSE_STRICT:
                raise RuntimeError(msg)
            print(msg+'; execução seguirá sem reuso.')
            return pd.DataFrame(),pd.DataFrame()

        raw['study_version']=STUDY_VERSION
        if not cls.empty and 'study_version' in cls.columns:
            cls['study_version']=STUDY_VERSION
        print(f'Reuso incremental ATIVO: {len(done)} condições completas reaproveitadas de {INCREMENTAL_BASE_RUN_ID}.')
        return raw,cls
    except Exception:
        if INCREMENTAL_REUSE_STRICT:
            raise
        print('Checkpoint incremental rejeitado; execução seguirá sem reuso.')
        return pd.DataFrame(),pd.DataFrame()

def execute_experiments(dataset, feature_context, plan):
    metrics_path=STUDY_DIR/'_internal_metrics_partial.csv'
    class_path=STUDY_DIR/'_internal_class_metrics_partial.csv'
    if metrics_path.is_file():
        raw_metrics=pd.read_csv(metrics_path)
        if 'study_version' not in raw_metrics or set(raw_metrics.study_version)!={STUDY_VERSION}:
            raw_metrics=pd.DataFrame()
        raw_class=pd.read_csv(class_path) if class_path.is_file() else pd.DataFrame()
    else:
        raw_metrics,raw_class=load_incremental_base(dataset,plan)
        if not raw_metrics.empty:
            raw_metrics.to_csv(metrics_path,index=False); raw_class.to_csv(class_path,index=False)
    completed=_completed_conditions(raw_metrics,dataset,plan)
    total=len(plan.registry)*len(plan.model_ids)+len(plan.special_experiments); current=0
    print(f'Plano incremental: {len(completed)} condição(ões) já completas; {total-len(completed)} condição(ões) pendentes nesta versão.')

    for entry in plan.registry:
        bundle=entry['bundle']; representation=entry['features']; family=entry['family']
        pending=[m for m in plan.model_ids if (representation,m) not in completed]
        if not pending:
            for model in plan.model_ids:
                current+=1
                print(f'[{current:03d}/{total}] {representation} × {model}: checkpoint')
            continue
        fold_arrays=[]; dims=[]
        for (tr,va),pack in zip(dataset.folds,feature_context.fold_packs):
            t0=time.perf_counter(); xtr=combine_blocks(pack['train'],bundle); xva=combine_blocks(pack['valid'],bundle); fusion_s=time.perf_counter()-t0
            feature_fit_s,feature_transform_s,components=bundle_component_time(pack,bundle)
            fold_arrays.append((tr,va,xtr,xva,feature_fit_s,feature_transform_s,fusion_s,components)); dims.append(xtr.shape[1])
        if len(set(dims))!=1:
            raise AssertionError(f'Dimensão instável em {representation}: {dims}')
        input_dim=dims[0]

        for model in plan.model_ids:
            current+=1
            if (representation,model) in completed:
                print(f'[{current:03d}/{total}] {representation} × {model}: checkpoint')
                continue
            rows=[]; crows=[]; condition_start=time.perf_counter()
            for fold,(tr,va,xtr,xva,feature_fit_s,feature_transform_s,fusion_s,components) in enumerate(fold_arrays,1):
                graph_s=0.0; pt=pe=None
                if plan.model_metadata.get(model,{}).get('role')=='relational':
                    tg=time.perf_counter(); p_train,p_eval,_=build_knn_propagation(xtr,xva,**plan.graph_config); graph_s=time.perf_counter()-tg
                    pt=torch.tensor(p_train,dtype=torch.float32,device=DEVICE).to_sparse().coalesce()
                    pe=torch.tensor(p_eval,dtype=torch.float32,device=DEVICE).to_sparse().coalesce()

                for seed in plan.model_seeds[model]:
                    if model in ('linear','mlp','gnn'):
                        prob,complexity,kind,train_s,predict_s=fit_predict_neural(xtr,dataset.targets[tr],xva,model,seed,pt,pe)
                    else:
                        prob,complexity,kind,train_s,predict_s=fit_predict_classical(xtr,dataset.targets[tr],xva,model,seed)
                    score=multilabel_metrics(dataset.targets[va],prob)
                    pipeline_s=feature_fit_s+feature_transform_s+fusion_s+graph_s+train_s+predict_s
                    rows.append({
                        'study_version':STUDY_VERSION,'family':family,'features':representation,'fontes':'|'.join(bundle),
                        'n_fontes':len(bundle),'input_dim':input_dim,'modelo':model,'fold':fold,'seed':seed,
                        'n_treino':len(tr),'n_validacao':len(va),'model_complexity':complexity,'complexity_kind':kind,
                        'feature_fit_seconds_incremental':feature_fit_s,'feature_transform_seconds_incremental':feature_transform_s,
                        'feature_fusion_seconds':fusion_s,'feature_components':'|'.join(components),'graph_build_seconds':graph_s,
                        'train_seconds':train_s,'predict_seconds':predict_s,'model_seconds':train_s+predict_s,
                        'pipeline_seconds_incremental':pipeline_s,**score,
                    })
                    cm=class_metrics(dataset.targets[va],prob,dataset.classes)
                    cm.insert(0,'seed',seed); cm.insert(0,'fold',fold); cm.insert(0,'modelo',model); cm.insert(0,'features',representation); cm.insert(0,'family',family)
                    crows.append(cm)
            wall=time.perf_counter()-condition_start
            for row in rows:
                row['condition_wall_seconds_observed']=wall
            raw_metrics=pd.concat([raw_metrics,pd.DataFrame(rows)],ignore_index=True)
            raw_class=pd.concat([raw_class,*crows],ignore_index=True) if crows else raw_class
            raw_metrics.to_csv(metrics_path,index=False); raw_class.to_csv(class_path,index=False)
            m=pd.DataFrame(rows)
            print(f'[{current:03d}/{total}] {representation} × {model} | {wall:6.1f}s | micro={m.f1_micro.mean():.3f} macro={m.f1_macro.mean():.3f}')


    # Experimentos focais especiais convergem para o mesmo contrato sem
    # expandir o produto cartesiano da matriz principal.
    for spec in plan.special_experiments:
        current+=1
        representation=str(spec['features']); model=str(spec['model_id']); family=str(spec['family'])
        adapter=str(spec.get('adapter',''))
        if (representation,model) in completed:
            print(f'[{current:03d}/{total}] {representation} × {model}: checkpoint')
            continue
        rows=[]; crows=[]; condition_start=time.perf_counter()

        if adapter=='homogeneous_operator':
            bundle=tuple(spec['bundle'])
            for fold,((tr,va),pack) in enumerate(zip(dataset.folds,feature_context.fold_packs),1):
                t0=time.perf_counter(); xtr=combine_blocks(pack['train'],bundle); xva=combine_blocks(pack['valid'],bundle); fusion_s=time.perf_counter()-t0
                feature_fit_s,feature_transform_s,components=bundle_component_time(pack,bundle)
                tg=time.perf_counter()
                edge_train=build_knn_edge_index(xtr,np.empty((0,xtr.shape[1]),dtype=np.float32),k=plan.graph_config['k'],min_similarity=plan.graph_config.get('min_similarity'))
                edge_eval=build_knn_edge_index(xtr,xva,k=plan.graph_config['k'],min_similarity=plan.graph_config.get('min_similarity'))
                graph_s=time.perf_counter()-tg
                for seed in plan.model_seeds[model]:
                    prob,complexity,kind,train_s,predict_s=fit_predict_homogeneous_operator(
                        xtr,dataset.targets[tr],xva,model,seed,edge_train,edge_eval
                    )
                    score=multilabel_metrics(dataset.targets[va],prob)
                    pipeline_s=feature_fit_s+feature_transform_s+fusion_s+graph_s+train_s+predict_s
                    rows.append({
                        'study_version':STUDY_VERSION,'family':family,'features':representation,'fontes':spec['fontes'],
                        'n_fontes':int(spec['n_fontes']),'input_dim':int(xtr.shape[1]),'modelo':model,'fold':fold,'seed':seed,
                        'n_treino':len(tr),'n_validacao':len(va),'model_complexity':complexity,'complexity_kind':kind,
                        'feature_fit_seconds_incremental':feature_fit_s,'feature_transform_seconds_incremental':feature_transform_s,
                        'feature_fusion_seconds':fusion_s,'feature_components':'|'.join(components),'graph_build_seconds':graph_s,
                        'train_seconds':train_s,'predict_seconds':predict_s,'model_seconds':train_s+predict_s,
                        'pipeline_seconds_incremental':pipeline_s,'graph_k':int(plan.graph_config['k']),**score,
                    })
                    cm=class_metrics(dataset.targets[va],prob,dataset.classes)
                    cm.insert(0,'seed',seed); cm.insert(0,'fold',fold); cm.insert(0,'modelo',model); cm.insert(0,'features',representation); cm.insert(0,'family',family)
                    crows.append(cm)

        elif adapter=='hetero_term':
            top_k=spec.get('top_k_terms')
            for fold,((tr,va),pack) in enumerate(zip(dataset.folds,feature_context.fold_packs),1):
                texts_tr=dataset.frame.iloc[tr].texto.tolist(); texts_va=dataset.frame.iloc[va].texto.tolist()
                # Vocabulário estrutural é ajustado apenas no treino externo.
                _,a_tr,a_va,fit_s,transform_s=_fit_tfidf_base(texts_tr,texts_va,WORD_CFGS['word_1'])
                tg=time.perf_counter()
                row_tr,col_tr,row_va,graph_meta=prepare_hetero_term_incidence(a_tr,a_va,top_k=top_k)
                graph_s=time.perf_counter()-tg
                xtr=pack['train']['minilm']; xva=pack['valid']['minilm']; input_dim=int(xtr.shape[1])
                for seed in plan.model_seeds[model]:
                    if model=='hetero_term_gnn':
                        prob,complexity,kind,train_s,predict_s=fit_predict_hetero_term(
                            xtr,dataset.targets[tr],xva,row_tr,col_tr,row_va,seed
                        )
                    elif model=='hetero_term_gatv2':
                        prob,complexity,kind,train_s,predict_s=fit_predict_hetero_term_gatv2(
                            xtr,dataset.targets[tr],xva,row_tr,col_tr,row_va,seed
                        )
                    else:
                        raise ValueError(f'Modelo heterogêneo sem adapter: {model}')
                    score=multilabel_metrics(dataset.targets[va],prob)
                    pipeline_s=fit_s+transform_s+graph_s+train_s+predict_s
                    rows.append({
                        'study_version':STUDY_VERSION,'family':family,'features':representation,'fontes':spec['fontes'],
                        'n_fontes':int(spec['n_fontes']),'input_dim':input_dim,'modelo':model,'fold':fold,'seed':seed,
                        'n_treino':len(tr),'n_validacao':len(va),'model_complexity':complexity,'complexity_kind':kind,
                        'feature_fit_seconds_incremental':fit_s,'feature_transform_seconds_incremental':transform_s,
                        'feature_fusion_seconds':0.0,'feature_components':'word_1_term_incidence',
                        'graph_build_seconds':graph_s,'train_seconds':train_s,'predict_seconds':predict_s,
                        'model_seconds':train_s+predict_s,'pipeline_seconds_incremental':pipeline_s,
                        'term_nodes':graph_meta['term_nodes'],'incidence_train':graph_meta['incidence_train'],
                        'incidence_valid':graph_meta['incidence_eval'],'zero_term_valid_rows':graph_meta['zero_term_eval_rows'],
                        'term_top_k':np.nan if graph_meta['term_top_k'] is None else int(graph_meta['term_top_k']),**score,
                    })
                    cm=class_metrics(dataset.targets[va],prob,dataset.classes)
                    cm.insert(0,'seed',seed); cm.insert(0,'fold',fold); cm.insert(0,'modelo',model); cm.insert(0,'features',representation); cm.insert(0,'family',family)
                    crows.append(cm)
        else:
            raise ValueError(f'Experimento especial sem adapter: {spec}')

        wall=time.perf_counter()-condition_start
        for row in rows: row['condition_wall_seconds_observed']=wall
        raw_metrics=pd.concat([raw_metrics,pd.DataFrame(rows)],ignore_index=True)
        raw_class=pd.concat([raw_class,*crows],ignore_index=True) if crows else raw_class
        raw_metrics.to_csv(metrics_path,index=False); raw_class.to_csv(class_path,index=False)
        m=pd.DataFrame(rows)
        suffix=''
        if 'term_nodes' in m:
            suffix=f" | termos≈{int(m.term_nodes.mean())} | incidências≈{int(m.incidence_train.mean())}"
        print(f'[{current:03d}/{total}] {representation} × {model} | {wall:6.1f}s | micro={m.f1_micro.mean():.3f} macro={m.f1_macro.mean():.3f}{suffix}')

    result=standardize_experiment_result(raw_metrics,raw_class,plan)
    persist_experiment_result(result,STUDY_DIR)
    expected=sum(_expected_rows_for_model(dataset,plan,m) for m in plan.model_ids)*len(plan.registry)
    expected+=sum(_expected_rows_for_model(dataset,plan,e['model_id']) for e in plan.special_experiments)
    assert len(raw_metrics)==expected,(len(raw_metrics),expected)
    return result


## 7.1 — Experimentos relacionais focais

A matriz principal continua pequena: Linear, MLP e **Fixed-kNN GNN** nas representações registradas; RF/XGBoost permanecem no catálogo com `enabled=False` durante a iteração.

Os novos modelos de grafo são executados como **experimentos focais**, não sobre todos os bundles.

### Bloco A — operadores homogêneos

Representação fixa: `MiniLM + word_1`. Topologia fixa: mesmo kNN indutivo (`k=5`) usado pelo baseline atual.

1. **MLP** — sem grafo (já existente).
2. **Fixed-kNN GNN** — baseline atual: `0.9 × self + 0.1 × média(vizinhos)` (já existente).
3. **GCN** — `GCNConv` em duas camadas.
4. **GraphSAGE** — `SAGEConv(mean)` em duas camadas.
5. **GATv2** — `GATv2Conv` em duas camadas.

Arestas kNN são direcionadas `vizinho_treino → obra_destino`. Na validação, nós de validação recebem mensagens apenas de treino; não existem arestas validação→treino nem validação→validação. Self-loops/root paths são os nativos de cada operador; o `self_weight=0.9` permanece específico do baseline Fixed-kNN.

**Pergunta:** dado o mesmo vetor e a mesma vizinhança, qual regra de message passing funciona melhor?

### Bloco B — heterogeneidade e hipótese de sinal difuso

O baseline heterogêneo permanece intacto: MiniLM nos nós obra; unigramas `word_1` como nós termo; incidência TF-IDF; estado dos termos construído exclusivamente a partir das obras de treino.

Depois são adicionadas ablações controladas:

- **top-30 termos/obra** por TF-IDF;
- **top-20 termos/obra**;
- **top-10 termos/obra**;
- **Hetero GATv2 termo→obra** com todos os termos: a construção train-only do estado dos termos é preservada, e somente a agregação termo→obra passa a aprender pesos de atenção. O TF-IDF é fornecido como atributo de aresta.

**Hipótese:** se a heterogênea atual sofre com sinal lexical difuso, reduzir conexões ou aprender quanto confiar em cada termo deve melhorar desempenho e/ou estabilidade.

Hypergraph continua planejado, mas desabilitado até esses dois blocos serem interpretados.

### Política incremental desta versão

Por padrão, a fase iterativa reutiliza as condições completas da run `20261025T142618Z_d9aa19` somente se dataset hash, split hash, CV, configuração de treino e configuração do grafo coincidirem. O modo é **estrito**: se o checkpoint não for localizado ou for incompatível, a execução para em vez de rerodar silenciosamente a matriz antiga. Nesta versão são esperadas 166 condições reutilizadas e apenas 7 novas. Para a rodada final congelada, executar com `GNN_INCREMENTAL_REUSE=0`.


## 8 — Diagnósticos experimentais opcionais

Esta camada recebe apenas os contratos de dados/features/resultados e produz diagnósticos **post-hoc**. Ela não altera treino, features ou arestas usadas pelos modelos.

Inclui:
- ganho relacional × homofilia;
- learning curve aninhada por grupo;
- amostras qualitativas de grafos (ego-graph, subgrafo por macrogênero e comparação da mesma amostra entre representações).

As visualizações de grafo usam rótulos verdadeiros **somente para colorir/anotar depois que o grafo já foi construído**; rótulos não entram na criação das arestas.

In [ ]:
def _metric_condition_summary(experiment_result):
    obs=experiment_result.metric_observations
    fold=(obs.groupby(['condition_id','metric','fold'],as_index=False).value.mean())
    return fold.groupby(['condition_id','metric'],as_index=False).agg(mean=('value','mean'),sd=('value','std'))


def run_relational_diagnostics(dataset, feature_context, experiment_result, plan):
    if not plan.diagnostics:
        return {}
    conditions=experiment_result.conditions
    summary=_metric_condition_summary(experiment_result)
    registry={r['features']:tuple(r['bundle']) for r in plan.registry}
    outputs={}

    for spec in plan.diagnostics:
        rel_model=spec['relational_model']; base_model=spec['baseline_model']; metric=spec['metric']
        available=set(conditions.model_id)
        if rel_model not in available or base_model not in available:
            continue

        edge_rows=[]
        for representation,bundle in registry.items():
            for fold,((tr,va),pack) in enumerate(zip(dataset.folds,feature_context.fold_packs),1):
                xtr=combine_blocks(pack['train'],bundle); xva=combine_blocks(pack['valid'],bundle)
                _,_,edges=build_knn_propagation(xtr,xva,**plan.graph_config)
                e=edges[edges.subset=='validacao']
                for row in e.itertuples(index=False):
                    yq=dataset.targets[va[int(row.destino_local)]].astype(bool)
                    yn=dataset.targets[tr[int(row.origem_treino_local)]].astype(bool)
                    inter=np.logical_and(yq,yn).sum(); union=np.logical_or(yq,yn).sum()
                    edge_rows.append({
                        'representation_name':representation,'fold':fold,'rank':int(row.rank),
                        'cosine':float(row.cosseno),'any_label_overlap':float(inter>0),
                        'jaccard':float(inter/union if union else 1.0),'shared_labels':int(inter),
                    })
        edges=pd.DataFrame(edge_rows)
        hom=(edges.groupby('representation_name',as_index=False).agg(
            mean_cosine=('cosine','mean'), any_label_overlap=('any_label_overlap','mean'),
            mean_jaccard=('jaccard','mean'), mean_shared_labels=('shared_labels','mean'),
        ))

        score=summary[summary.metric==metric].merge(conditions[['condition_id','representation_name','model_id']],on='condition_id',validate='many_to_one')
        rel=score[score.model_id==rel_model][['representation_name','mean']].rename(columns={'mean':'relational_score'})
        base=score[score.model_id==base_model][['representation_name','mean']].rename(columns={'mean':'baseline_score'})
        view=hom.merge(rel,on='representation_name').merge(base,on='representation_name')
        view['delta']=view.relational_score-view.baseline_score
        view['diagnostic_id']=spec['id']
        view['comparison_label']=f"{plan.model_metadata[rel_model]['display_name']} − {plan.model_metadata[base_model]['display_name']}"
        view['metric']=metric
        outputs[spec['id']]=view
        outputs[f"{spec['id']}__edges"]=edges
    return outputs


LEARNING_CURVE_CONFIG = {
    'id':'learning_curve',
    'version':'learning_curve_nested_group_balanced_v1',
    'representation_name':'minilm+word_1',
    'bundle':('minilm','word_1'),
    'models':('mlp','gnn'),
    'fractions':(0.20,0.40,0.60,0.80,1.00),
    'subset_seed':CV_SEED + 1000,
    'subset_policy':'nested_group_greedy_multilabel_balance_train_only',
    'validation_policy':'same frozen outer validation fold for every fraction',
    'feature_policy':'MiniLM frozen; word_1 TF-IDF/SVD refit on each training subset',
}


def _learning_curve_group_order(dataset, tr, seed):
    tr=np.asarray(tr,dtype=int)
    group_values=dataset.frame.iloc[tr].grupo_franquia.astype(str).to_numpy()
    full_counts=dataset.targets[tr].sum(0).astype(float)
    full_prev=full_counts/max(len(tr),1)
    weights=1/np.sqrt(np.maximum(full_prev,1/max(len(tr),1)))
    weights=weights/max(weights.mean(),1e-12)

    groups=[]
    for group in sorted(np.unique(group_values)):
        idx=tr[group_values==group]
        groups.append({
            'group':str(group), 'indices':idx,
            'counts':dataset.targets[idx].sum(0).astype(float), 'n':int(len(idx)),
        })
    rng=np.random.default_rng(seed)
    tie={g['group']:float(rng.random()) for g in groups}
    remaining=list(groups); order=[]
    selected_counts=np.zeros(dataset.targets.shape[1],dtype=float); selected_n=0

    while remaining:
        best_idx=None; best_score=None
        for i,item in enumerate(remaining):
            new_counts=selected_counts+item['counts']; new_n=selected_n+item['n']
            prevalence=new_counts/max(new_n,1)
            distribution_error=float(np.mean(weights*np.abs(prevalence-full_prev)))
            missing_penalty=float(np.mean((new_counts==0)&(full_counts>0)))
            score=distribution_error + 0.35*missing_penalty + 1e-9*tie[item['group']]
            if best_score is None or score<best_score:
                best_idx=i; best_score=score
        item=remaining.pop(best_idx)
        order.append(item); selected_counts+=item['counts']; selected_n+=item['n']
    return order


def _learning_curve_nested_subsets(dataset, tr, fractions, seed):
    order=_learning_curve_group_order(dataset,tr,seed)
    selected=[]; cursor=0; outputs=[]
    n_outer=len(tr)
    for fraction in fractions:
        target=n_outer if float(fraction)>=1 else int(np.ceil(float(fraction)*n_outer))
        while cursor<len(order):
            support=dataset.targets[np.asarray(selected,dtype=int)].sum(0) if selected else np.zeros(dataset.targets.shape[1])
            if len(selected)>=target and np.all(support>0):
                break
            selected.extend(order[cursor]['indices'].tolist()); cursor+=1
        subset=np.asarray(sorted(selected),dtype=int)
        support=dataset.targets[subset].sum(0)
        if np.any(support==0):
            raise RuntimeError(f'Learning curve sem suporte para alguma classe em {fraction:.0%}.')
        outputs.append({
            'target_fraction':float(fraction), 'fraction_percent':int(round(100*float(fraction))),
            'actual_fraction':float(len(subset)/n_outer), 'indices':subset,
            'n_train':int(len(subset)),
            'n_groups':int(dataset.frame.iloc[subset].grupo_franquia.nunique()),
            'support':support.astype(int),
        })
    for a,b in zip(outputs,outputs[1:]):
        assert set(a['indices']).issubset(set(b['indices']))
    return outputs


def _learning_curve_features(dataset, feature_context, fold_index, tr, va, subset_indices):
    pack=feature_context.fold_packs[fold_index]
    outer_position={int(idx):i for i,idx in enumerate(np.asarray(tr,dtype=int))}
    local=np.asarray([outer_position[int(idx)] for idx in subset_indices],dtype=int)
    minilm_tr=pack['train']['minilm'][local]
    minilm_va=pack['valid']['minilm']

    texts_tr=dataset.frame.iloc[subset_indices].texto.tolist()
    texts_va=dataset.frame.iloc[va].texto.tolist()
    _,a_tr,a_va,fit_base,transform_base=_fit_tfidf_base(texts_tr,texts_va,WORD_CFGS['word_1'])
    word_tr,word_va,_,fit_svd,transform_svd=_fit_svd(a_tr,a_va,LEX_SVD_DIM,seed=CV_SEED)

    t0=time.perf_counter()
    xtr=combine_blocks({'minilm':minilm_tr,'word_1':word_tr},('minilm','word_1'))
    xva=combine_blocks({'minilm':minilm_va,'word_1':word_va},('minilm','word_1'))
    fusion_s=time.perf_counter()-t0
    return xtr,xva,{
        'feature_fit_seconds':float(fit_base+fit_svd),
        'feature_transform_seconds':float(transform_base+transform_svd),
        'feature_fusion_seconds':float(fusion_s),
        'word_1_vocab':int(a_tr.shape[1]), 'word_1_dim':int(word_tr.shape[1]),
    }


def run_learning_curve_diagnostic(dataset, feature_context, experiment_result, plan, spec=LEARNING_CURVE_CONFIG):
    models=tuple(m for m in spec['models'] if m in plan.model_ids)
    if not models:
        return {}
    for model in models:
        if model not in ('mlp','gnn'):
            raise ValueError('Learning curve atual usa a interface neural MLP/GNN.')

    metrics_path=STUDY_DIR/'learning_curve_metrics_partial.csv'
    class_path=STUDY_DIR/'learning_curve_class_metrics_partial.csv'
    if metrics_path.is_file():
        raw=pd.read_csv(metrics_path)
        if 'diag_version' not in raw or set(raw.diag_version)!={spec['version']}:
            raw=pd.DataFrame()
    else:
        raw=pd.DataFrame()
    if class_path.is_file():
        raw_class=pd.read_csv(class_path)
        if 'diag_version' not in raw_class or set(raw_class.diag_version)!={spec['version']}:
            raw_class=pd.DataFrame()
    else:
        raw_class=pd.DataFrame()

    done=set()
    if not raw.empty:
        done=set(zip(raw.fold.astype(int),raw.fraction_percent.astype(int),raw.model_id.astype(str),raw.seed.astype(int)))

    subset_rows=[]; support_rows=[]
    total_units=len(dataset.folds)*len(spec['fractions']); unit=0
    for fold_index,(tr,va) in enumerate(dataset.folds):
        subsets=_learning_curve_nested_subsets(dataset,tr,spec['fractions'],spec['subset_seed']+fold_index)
        for subset in subsets:
            unit+=1; pct=subset['fraction_percent']
            for class_name,support in zip(dataset.classes,subset['support']):
                support_rows.append({
                    'fold':fold_index+1,'target_fraction':subset['target_fraction'],'fraction_percent':pct,
                    'actual_fraction':subset['actual_fraction'],'n_train':subset['n_train'],
                    'class_name':class_name,'train_support':int(support),
                })
            subset_rows.append({
                'fold':fold_index+1,'target_fraction':subset['target_fraction'],'fraction_percent':pct,
                'actual_fraction':subset['actual_fraction'],'n_train':subset['n_train'],'n_groups':subset['n_groups'],
            })

            expected_keys=[(fold_index+1,pct,m,int(seed)) for m in models for seed in plan.model_seeds[m]]
            if all(k in done for k in expected_keys):
                print(f"[LC {unit:02d}/{total_units}] fold {fold_index+1} | {pct:3d}% | checkpoint")
                continue

            xtr,xva,feature_meta=_learning_curve_features(dataset,feature_context,fold_index,tr,va,subset['indices'])
            pt=pe=None; graph_s=0.0
            if any(plan.model_metadata.get(m,{}).get('role')=='relational' for m in models):
                tg=time.perf_counter(); p_train,p_eval,_=build_knn_propagation(xtr,xva,**plan.graph_config); graph_s=time.perf_counter()-tg
                pt=torch.tensor(p_train,dtype=torch.float32,device=DEVICE).to_sparse().coalesce()
                pe=torch.tensor(p_eval,dtype=torch.float32,device=DEVICE).to_sparse().coalesce()

            new_rows=[]; new_class=[]
            for model in models:
                use_graph=plan.model_metadata.get(model,{}).get('role')=='relational'
                for seed in plan.model_seeds[model]:
                    key=(fold_index+1,pct,model,int(seed))
                    if key in done:
                        continue
                    prob,complexity,kind,train_s,predict_s=fit_predict_neural(
                        xtr,dataset.targets[subset['indices']],xva,model,int(seed),pt if use_graph else None,pe if use_graph else None
                    )
                    score=multilabel_metrics(dataset.targets[va],prob)
                    row={
                        'diag_version':spec['version'],'representation_name':spec['representation_name'],
                        'fold':fold_index+1,'target_fraction':subset['target_fraction'],'fraction_percent':pct,
                        'actual_fraction':subset['actual_fraction'],'n_train':subset['n_train'],'n_groups':subset['n_groups'],
                        'model_id':model,'seed':int(seed),'input_dim':int(xtr.shape[1]),
                        'word_1_vocab':feature_meta['word_1_vocab'],'word_1_dim':feature_meta['word_1_dim'],
                        'feature_fit_seconds':feature_meta['feature_fit_seconds'],
                        'feature_transform_seconds':feature_meta['feature_transform_seconds'],
                        'feature_fusion_seconds':feature_meta['feature_fusion_seconds'],
                        'graph_build_seconds':graph_s if use_graph else 0.0,
                        'train_seconds':float(train_s),'predict_seconds':float(predict_s),
                        **score,
                    }
                    new_rows.append(row)
                    cm=class_metrics(dataset.targets[va],prob,dataset.classes)
                    cm['diag_version']=spec['version']; cm['representation_name']=spec['representation_name']
                    cm['fold']=fold_index+1; cm['target_fraction']=subset['target_fraction']; cm['fraction_percent']=pct
                    cm['actual_fraction']=subset['actual_fraction']; cm['n_train']=subset['n_train']; cm['model_id']=model; cm['seed']=int(seed)
                    new_class.append(cm)
                    done.add(key)

            if new_rows:
                raw=pd.concat([raw,pd.DataFrame(new_rows)],ignore_index=True)
                raw_class=pd.concat([raw_class,*new_class],ignore_index=True)
                raw.to_csv(metrics_path,index=False); raw_class.to_csv(class_path,index=False)
            print(f"[LC {unit:02d}/{total_units}] fold {fold_index+1} | {pct:3d}% | n={subset['n_train']} | pronto")

    expected=len(dataset.folds)*len(spec['fractions'])*sum(len(plan.model_seeds[m]) for m in models)
    if len(raw)!=expected:
        raise AssertionError(f'Learning curve incompleta: {len(raw)} linhas; esperado {expected}.')

    # O ponto de 100% deve reproduzir a mesma condição da matriz principal.
    # Isso valida que a learning curve não mudou silenciosamente o pipeline.
    if experiment_result is not None and not experiment_result.raw_metrics.empty:
        base=experiment_result.raw_metrics[
            (experiment_result.raw_metrics.features.astype(str)==str(spec['representation_name'])) &
            (experiment_result.raw_metrics.modelo.astype(str).isin(models))
        ].copy()
        lc100=raw[raw.fraction_percent==100].copy()
        if not base.empty and not lc100.empty:
            merged=lc100.merge(
                base[['modelo','fold','seed',*plan.primary_metrics]],
                left_on=['model_id','fold','seed'],right_on=['modelo','fold','seed'],
                suffixes=('_lc','_main'),validate='one_to_one'
            )
            for metric in plan.primary_metrics:
                diff=np.abs(merged[f'{metric}_lc']-merged[f'{metric}_main'])
                if len(diff) and float(diff.max())>1e-7:
                    raise AssertionError(f'Learning curve 100% divergiu da matriz principal em {metric}: {float(diff.max())}')
            print('Learning curve 100% reproduz a matriz principal: OK')

    raw.to_csv(STUDY_DIR/'learning_curve_metrics.csv',index=False)
    raw_class.to_csv(STUDY_DIR/'learning_curve_class_metrics.csv',index=False)
    subsets_df=pd.DataFrame(subset_rows).drop_duplicates().sort_values(['fold','fraction_percent'])
    support_df=pd.DataFrame(support_rows).drop_duplicates().sort_values(['fold','fraction_percent','class_name'])
    subsets_df.to_csv(STUDY_DIR/'learning_curve_subsets.csv',index=False)
    support_df.to_csv(STUDY_DIR/'learning_curve_subset_support.csv',index=False)

    metric_long=raw.melt(
        id_vars=['fold','fraction_percent','target_fraction','actual_fraction','n_train','model_id'],
        value_vars=[m for m in plan.primary_metrics if m in raw.columns],var_name='metric',value_name='value'
    )
    fold_metric=(metric_long.groupby(['model_id','fraction_percent','metric','fold'],as_index=False).agg(
        value=('value','mean'),n_train=('n_train','first'),actual_fraction=('actual_fraction','first')
    ))
    global_summary=(fold_metric.groupby(['model_id','fraction_percent','metric'],as_index=False).agg(
        mean=('value','mean'),sd=('value','std'),n_train_mean=('n_train','mean'),
        n_train_min=('n_train','min'),n_train_max=('n_train','max'),actual_fraction_mean=('actual_fraction','mean')
    ))
    global_summary['model_name']=global_summary.model_id.map(lambda m: plan.model_metadata.get(m,{}).get('display_name',m))

    class_long=raw_class.melt(
        id_vars=['fold','fraction_percent','target_fraction','actual_fraction','n_train','model_id','class_name'],
        value_vars=[m for m in ('precision','recall','f1') if m in raw_class.columns],var_name='metric',value_name='value'
    )
    class_fold=(class_long.groupby(['model_id','fraction_percent','class_name','metric','fold'],as_index=False).agg(value=('value','mean')))
    class_summary=(class_fold.groupby(['model_id','fraction_percent','class_name','metric'],as_index=False).agg(
        mean=('value','mean'),sd=('value','std')
    ))
    class_summary['model_name']=class_summary.model_id.map(lambda m: plan.model_metadata.get(m,{}).get('display_name',m))

    primary0,primary1=plan.primary_metrics[:2]
    micro=global_summary[global_summary.metric==primary0].copy()
    macro=global_summary[global_summary.metric==primary1].copy()
    focus_model=next((m for m in models if plan.model_metadata.get(m,{}).get('role')=='relational'),models[0])
    heat_src=class_summary[(class_summary.model_id==focus_model)&(class_summary.metric=='f1')]
    heat=heat_src.pivot(index='class_name',columns='fraction_percent',values='mean').reindex(index=list(dataset.classes))
    heat=heat.reindex(columns=sorted(heat.columns)); heat.columns=[f'{int(c)}%' for c in heat.columns]

    f1_focus=heat_src.pivot(index='class_name',columns='fraction_percent',values='mean')
    low=min(spec['fractions']); high=max(spec['fractions']); low_pct=int(round(low*100)); high_pct=int(round(high*100))
    gain=pd.DataFrame({'class_name':list(dataset.classes)})
    gain=gain.merge(f1_focus[[low_pct,high_pct]].rename(columns={low_pct:'f1_low',high_pct:'f1_high'}).reset_index(),on='class_name',how='left')
    gain['delta_f1']=gain.f1_high-gain.f1_low
    gain['support_full']=dataset.targets.sum(0).astype(int)
    gain['model_id']=focus_model; gain['model_name']=plan.model_metadata.get(focus_model,{}).get('display_name',focus_model)
    gain=gain.sort_values('delta_f1').reset_index(drop=True)

    return {
        'learning_curve':global_summary,
        'learning_curve__micro':micro,
        'learning_curve__macro':macro,
        'learning_curve__by_class':class_summary,
        'learning_curve__class_heatmap':heat,
        'learning_curve__class_gain':gain,
        'learning_curve__subsets':subsets_df,
        'learning_curve__subset_support':support_df,
    }




GRAPH_VIS_CONFIG = {
    'id':'graph_samples',
    'version':'graph_samples_fold_exact_v1',
    'representative_fold_policy':'closest_reference_fold_to_global_primary_metrics',
    'ego_center_policy':'median_validation_mean_cosine_without_labels',
    'ego_second_hop_per_neighbor':2,
    'max_ego_nodes':18,
    'class_policy':'median_global_support_with_validation_presence',
    'class_seed_nodes':6,
    'class_neighbors_per_seed':3,
    'max_class_nodes':24,
    'comparison_policy':'reference_bundle_components_plus_reference',
    'layout_policy':'PCA_on_reference_representation_shared_across_comparison_views',
    'labels_policy':'posthoc_visual_annotation_only',
}


def _reference_condition_for_diagnostics(experiment_result, allowed_representations=None):
    # Mesma heurística usada pelo assembler, aplicada ao contrato de resultados.
    obs=experiment_result.metric_observations
    by_fold=obs.groupby(['condition_id','metric','fold'],as_index=False).value.mean()
    summary=by_fold.groupby(['condition_id','metric'],as_index=False).agg(mean=('value','mean'),sd=('value','std'))
    mean_w=summary.pivot(index='condition_id',columns='metric',values='mean').add_suffix('_mean')
    sd_w=summary.pivot(index='condition_id',columns='metric',values='sd').add_suffix('_sd')
    x=experiment_result.conditions.merge(mean_w.reset_index(),on='condition_id').merge(sd_w.reset_index(),on='condition_id')
    policy=experiment_result.metadata['reference_policy']; metrics=list(policy['metrics'])
    vals=np.column_stack([x[f'{m}_mean'].clip(lower=0).to_numpy(float) for m in metrics])
    denom=np.sum(np.divide(1.0,vals,out=np.full_like(vals,np.inf),where=vals>0),axis=1)
    hmean=np.divide(len(metrics),denom,out=np.zeros(len(x),dtype=float),where=np.isfinite(denom)&(denom>0))
    sd_sum=np.zeros(len(x),dtype=float)
    for m in metrics:
        if f'{m}_sd' in x: sd_sum+=x[f'{m}_sd'].fillna(0).to_numpy(float)
    x['reference_score']=hmean-float(policy.get('stability_weight',0))*sd_sum
    if allowed_representations is not None:
        allowed=set(map(str,allowed_representations))
        x=x[x.representation_name.astype(str).isin(allowed)].copy()
        if x.empty: return {}
    ranking=x.sort_values(['reference_score',*[f'{m}_mean' for m in reversed(metrics)]],ascending=False).reset_index(drop=True)
    return ranking.iloc[0].to_dict()


def _representative_fold_for_condition(experiment_result, condition_id, primary_metrics):
    obs=experiment_result.metric_observations
    q=obs[(obs.condition_id==condition_id)&(obs.metric.isin(primary_metrics))]
    if q.empty: return 0
    fold=q.groupby(['fold','metric'],as_index=False).value.mean().pivot(index='fold',columns='metric',values='value')
    target=q.groupby('metric').value.mean()
    cols=[m for m in primary_metrics if m in fold.columns and m in target.index]
    if not cols: return int(fold.index.min())-1
    dist=np.zeros(len(fold),dtype=float)
    for m in cols: dist+=(fold[m].to_numpy(float)-float(target[m]))**2
    return int(fold.index[int(np.argmin(dist))])-1


def _globalize_graph_edges(edges, tr, va):
    rows=[]
    for r in edges.itertuples(index=False):
        src=int(tr[int(r.origem_treino_local)])
        if str(r.subset)=='validacao': dst=int(va[int(r.destino_local)])
        else: dst=int(tr[int(r.destino_local)])
        rows.append({'source_index':src,'target_index':dst,'rank':int(r.rank),'cosine':float(r.cosseno),'subset':str(r.subset)})
    return pd.DataFrame(rows)


def _features_for_global_indices(pack, bundle, tr, va, global_indices):
    xtr=combine_blocks(pack['train'],bundle); xva=combine_blocks(pack['valid'],bundle)
    tr_map={int(g):i for i,g in enumerate(tr)}; va_map={int(g):i for i,g in enumerate(va)}
    rows=[]
    for g in global_indices:
        g=int(g)
        if g in tr_map: rows.append(xtr[tr_map[g]])
        elif g in va_map: rows.append(xva[va_map[g]])
        else: raise KeyError(f'Índice global fora do fold: {g}')
    return np.asarray(rows,dtype=np.float32)


def _shared_layout(reference_features, center_pos=None):
    x=np.asarray(reference_features,dtype=np.float32)
    if len(x)==1: return np.array([[0.0,0.0]],dtype=np.float32)
    from sklearn.decomposition import PCA
    if np.allclose(x,x[0]):
        ang=np.linspace(0,2*np.pi,len(x),endpoint=False)
        coords=np.column_stack([np.cos(ang),np.sin(ang)]).astype(np.float32)
    else:
        ncomp=min(2,len(x),x.shape[1])
        coords=PCA(n_components=ncomp).fit_transform(x).astype(np.float32)
        if ncomp<2: coords=np.column_stack([coords,np.zeros(len(coords),dtype=np.float32)])
        scale=np.max(np.abs(coords),axis=0); scale[scale<1e-8]=1.0; coords/=scale
    if center_pos is not None:
        coords-=coords[int(center_pos)]
    return coords


def _node_label_names(dataset, idx):
    return [str(c) for c,v in zip(dataset.classes,dataset.targets[int(idx)]) if float(v)>0.5]


def _jaccard_rows(a,b):
    aa=np.asarray(a,dtype=bool); bb=np.asarray(b,dtype=bool)
    union=np.logical_or(aa,bb).sum(); inter=np.logical_and(aa,bb).sum()
    return float(inter/union if union else 1.0)


def _graph_sample_nodes(dataset, indices, tr, va, coords, sample_id, representation_name,
                        center_index=None, focus_class=None, label_indices=()):
    va_set=set(map(int,va)); label_indices=set(map(int,label_indices))
    center_y=dataset.targets[int(center_index)] if center_index is not None else None
    rows=[]
    for pos,g in enumerate(map(int,indices)):
        labels=_node_label_names(dataset,g)
        name=str(dataset.frame.iloc[g].Nome) if 'Nome' in dataset.frame.columns else str(dataset.frame.iloc[g].Id)
        row={
            'sample_id':sample_id,'representation_name':representation_name,'node_index':g,
            'node_id':str(dataset.frame.iloc[g].Id),'display_name':name,'split':'validação' if g in va_set else 'treino',
            'x':float(coords[pos,0]),'y':float(coords[pos,1]),'is_center':bool(center_index is not None and g==int(center_index)),
            'show_label':bool((center_index is not None and g==int(center_index)) or g in label_indices),
            'labels':' | '.join(labels),'n_labels':len(labels),
        }
        if center_y is not None: row['label_jaccard_to_center']=_jaccard_rows(dataset.targets[g],center_y)
        if focus_class is not None:
            j=list(dataset.classes).index(focus_class)
            row['focus_class_present']=bool(dataset.targets[g,j]>0.5)
            row['focus_group']=focus_class if row['focus_class_present'] else 'Outros'
        rows.append(row)
    return pd.DataFrame(rows)


def _sample_edges_for_representation(pack,bundle,tr,va,selected_indices,graph_cfg,sample_id,representation_name):
    xtr=combine_blocks(pack['train'],bundle); xva=combine_blocks(pack['valid'],bundle)
    _,_,edges=build_knn_propagation(xtr,xva,**graph_cfg)
    ge=_globalize_graph_edges(edges,tr,va)
    keep=set(map(int,selected_indices))
    if ge.empty:
        return pd.DataFrame(columns=['sample_id','representation_name','source_index','target_index','rank','cosine','subset'])
    ge=ge[ge.source_index.isin(keep)&ge.target_index.isin(keep)].copy()
    ge.insert(0,'representation_name',representation_name); ge.insert(0,'sample_id',sample_id)
    return ge.reset_index(drop=True)


def run_graph_visualization_diagnostic(dataset, feature_context, experiment_result, plan, spec=GRAPH_VIS_CONFIG):
    if experiment_result is None or experiment_result.conditions.empty:
        return {}
    registry={r['features']:tuple(r['bundle']) for r in plan.registry}
    # Se um experimento estrutural especial vencer o ranking global, a visualização
    # homogênea usa a melhor representação pertencente à matriz principal.
    reference=_reference_condition_for_diagnostics(experiment_result,allowed_representations=registry)
    if not reference: return {}
    ref_name=str(reference['representation_name'])
    ref_bundle=registry[ref_name]
    fold_index=_representative_fold_for_condition(experiment_result,reference['condition_id'],plan.primary_metrics)
    tr,va=dataset.folds[fold_index]; pack=feature_context.fold_packs[fold_index]

    xtr=combine_blocks(pack['train'],ref_bundle); xva=combine_blocks(pack['valid'],ref_bundle)
    _,_,edges_ref=build_knn_propagation(xtr,xva,**plan.graph_config)
    ge_ref=_globalize_graph_edges(edges_ref,tr,va)
    valid_edges=edges_ref[edges_ref.subset=='validacao']
    if valid_edges.empty: return {}

    # Centro do ego-graph: conectividade típica, sem consultar rótulos.
    qmean=valid_edges.groupby('destino_local').cosseno.mean().sort_index()
    med=float(qmean.median()); q=int((qmean-med).abs().sort_values(kind='stable').index[0])
    center=int(va[q])
    first=valid_edges[valid_edges.destino_local==q].sort_values('rank')
    first_globals=[int(tr[int(j)]) for j in first.origem_treino_local]
    selected=[center,*first_globals]
    train_edges=edges_ref[edges_ref.subset=='treino']
    for g in first_globals:
        local=np.where(tr==g)[0]
        if not len(local): continue
        sec=train_edges[train_edges.destino_local==int(local[0])].sort_values('rank').head(int(spec['ego_second_hop_per_neighbor']))
        selected.extend(int(tr[int(j)]) for j in sec.origem_treino_local)
    selected=list(dict.fromkeys(selected))[:int(spec['max_ego_nodes'])]
    first_label=set(first_globals)
    ref_feat=_features_for_global_indices(pack,ref_bundle,tr,va,selected)
    coords=_shared_layout(ref_feat,center_pos=selected.index(center))

    nodes_all=[]; edges_all=[]; sample_rows=[]
    # Referência + componentes individuais do bundle, todos com os mesmos nós/layout.
    comparison=[]
    for src in ref_bundle:
        if src in pack['train']: comparison.append((str(src),(src,)))
    comparison.append((ref_name,ref_bundle))
    seen=set(); comparison=[x for x in comparison if not (x[0] in seen or seen.add(x[0]))]

    for rep_name,bundle in comparison:
        sample_id='graph_ego_reference' if rep_name==ref_name else f"graph_compare__{re.sub(r'[^a-zA-Z0-9_]+','_',rep_name)}"
        nodes=_graph_sample_nodes(dataset,selected,tr,va,coords,sample_id,rep_name,center_index=center,label_indices=first_label)
        ed=_sample_edges_for_representation(pack,bundle,tr,va,selected,plan.graph_config,sample_id,rep_name)
        nodes_all.append(nodes); edges_all.append(ed)
        sample_rows.append({
            'sample_id':sample_id,'view_type':'ego' if rep_name==ref_name else 'comparison',
            'representation_name':rep_name,'fold':fold_index+1,'center_id':str(dataset.frame.iloc[center].Id),
            'center_name':str(dataset.frame.iloc[center].Nome),'focus_class':'',
            'n_nodes':len(nodes),'n_edges':len(ed),
            'title':('Ego-graph' if rep_name==ref_name else 'Mesma amostra')+f' — {rep_name}',
            'color_mode':'center_jaccard','note':'Layout compartilhado da representação de referência; cor = Jaccard de rótulos com o centro (post-hoc).',
        })

    # Subgrafo por classe: classe de suporte mediano com presença no fold representativo.
    support=dataset.targets.sum(0).astype(int); median=float(np.median(support))
    candidates=sorted(range(len(dataset.classes)),key=lambda j:(abs(int(support[j])-median),str(dataset.classes[j])))
    focus_j=next((j for j in candidates if dataset.targets[va,j].sum()>0),candidates[0]); focus_class=str(dataset.classes[focus_j])
    positives=[int(g) for g in va if dataset.targets[int(g),focus_j]>0.5]
    positives=sorted(positives,key=lambda g:str(dataset.frame.iloc[g].Id))[:int(spec['class_seed_nodes'])]
    class_selected=list(positives)
    for g in positives:
        qlocal=int(np.where(va==g)[0][0])
        nbr=valid_edges[valid_edges.destino_local==qlocal].sort_values('rank').head(int(spec['class_neighbors_per_seed']))
        class_selected.extend(int(tr[int(j)]) for j in nbr.origem_treino_local)
    class_selected=list(dict.fromkeys(class_selected))[:int(spec['max_class_nodes'])]
    class_feat=_features_for_global_indices(pack,ref_bundle,tr,va,class_selected)
    class_coords=_shared_layout(class_feat)
    class_id='graph_class_reference'
    class_nodes=_graph_sample_nodes(dataset,class_selected,tr,va,class_coords,class_id,ref_name,focus_class=focus_class,label_indices=positives)
    class_edges=_sample_edges_for_representation(pack,ref_bundle,tr,va,class_selected,plan.graph_config,class_id,ref_name)
    nodes_all.append(class_nodes); edges_all.append(class_edges)
    sample_rows.append({
        'sample_id':class_id,'view_type':'class','representation_name':ref_name,'fold':fold_index+1,
        'center_id':'','center_name':'','focus_class':focus_class,'n_nodes':len(class_nodes),'n_edges':len(class_edges),
        'title':f'Subgrafo por macrogênero — {focus_class} — {ref_name}',
        'color_mode':'focus_class','note':'Seeds de validação da classe + vizinhos de treino; rótulos usados apenas para seleção/coloração post-hoc.',
    })

    nodes_df=pd.concat(nodes_all,ignore_index=True) if nodes_all else pd.DataFrame()
    edges_df=pd.concat(edges_all,ignore_index=True) if edges_all else pd.DataFrame()
    samples_df=pd.DataFrame(sample_rows)
    return {
        'graph_samples__samples':samples_df,
        'graph_samples__nodes':nodes_df,
        'graph_samples__edges':edges_df,
    }



def _focused_condition_rows(experiment_result, wanted):
    cond=experiment_result.conditions; obs=experiment_result.metric_observations
    rows=[]
    for representation,model,label,role in wanted:
        q=cond[(cond.representation_name==representation)&(cond.model_id==model)]
        if q.empty: continue
        cid=str(q.iloc[0].condition_id); o=obs[obs.condition_id==cid]
        row={'condition_id':cid,'condition_label':label,'comparison_role':role,'representation_name':representation,'model_id':model}
        for metric in ('f1_micro','f1_macro'):
            vals=o[o.metric==metric].groupby('fold').value.mean()
            row[f'{metric}_mean']=float(vals.mean()) if len(vals) else np.nan
            row[f'{metric}_sd']=float(vals.std()) if len(vals)>1 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def run_graph_operator_diagnostic(experiment_result):
    """Mesma representação e mesmo kNN; muda apenas o operador de message passing."""
    wanted=[
        ('minilm+word_1','mlp','MLP','individual_vector'),
        ('minilm+word_1','gnn','Fixed-kNN GNN','fixed_message_passing'),
        ('minilm+word_1','gcn','GCN','gcn'),
        ('minilm+word_1','graphsage','GraphSAGE','graphsage'),
        ('minilm+word_1','gatv2','GATv2','attention'),
    ]
    comparison=_focused_condition_rows(experiment_result,wanted)
    if not comparison.empty:
        base=comparison[comparison.model_id=='mlp']
        if not base.empty:
            for metric in ('f1_micro','f1_macro'):
                comparison[f'delta_{metric}_vs_mlp']=comparison[f'{metric}_mean']-float(base.iloc[0][f'{metric}_mean'])
    return {'graph_operator_comparison':comparison}


def run_hetero_structural_diagnostic(experiment_result):
    """Vetor, grafo homogêneo e ablações heterogêneas da hipótese de difusão."""
    base=HETERO_TERM_CFG['representation_name']
    wanted=[
        ('minilm+word_1','mlp','MLP — MiniLM+word_1','individual_vector'),
        ('minilm+word_1','gnn','Fixed-kNN GNN — MiniLM+word_1','homogeneous_graph'),
        (base,'hetero_term_gnn','Hetero — todos os termos','heterogeneous_all_terms'),
        (f'{base}_top30','hetero_term_gnn','Hetero — top-30 TF-IDF','heterogeneous_topk'),
        (f'{base}_top20','hetero_term_gnn','Hetero — top-20 TF-IDF','heterogeneous_topk'),
        (f'{base}_top10','hetero_term_gnn','Hetero — top-10 TF-IDF','heterogeneous_topk'),
        (base,'hetero_term_gatv2','Hetero GATv2 — todos os termos','heterogeneous_attention'),
    ]
    comparison=_focused_condition_rows(experiment_result,wanted)
    if not comparison.empty:
        base_row=comparison[comparison.comparison_role=='individual_vector']
        if not base_row.empty:
            for metric in ('f1_micro','f1_macro'):
                b=float(base_row.iloc[0][f'{metric}_mean'])
                comparison[f'delta_{metric}_vs_mlp']=comparison[f'{metric}_mean']-b

    raw=experiment_result.raw_metrics
    structure_cols=['features','modelo','fold','term_nodes','incidence_train','incidence_valid','zero_term_valid_rows','term_top_k']
    structure=pd.DataFrame()
    if not raw.empty and all(c in raw.columns for c in structure_cols):
        q=raw[raw.modelo.isin(['hetero_term_gnn','hetero_term_gatv2'])].copy()
        if not q.empty:
            structure=q[structure_cols+['n_treino']].drop_duplicates().sort_values(['features','modelo','fold']).reset_index(drop=True)
            structure['mean_terms_per_train_work']=structure.incidence_train/np.maximum(1,structure.n_treino)
            structure=structure.drop(columns='n_treino')
    return {'hetero_structural_comparison':comparison,'hetero_term_structure':structure}


def run_diagnostics(dataset, feature_context, experiment_result, plan):
    diagnostics={}
    diagnostics.update(run_relational_diagnostics(dataset,feature_context,experiment_result,plan))
    diagnostics.update(run_graph_operator_diagnostic(experiment_result))
    diagnostics.update(run_hetero_structural_diagnostic(experiment_result))
    diagnostics.update(run_learning_curve_diagnostic(dataset,feature_context,experiment_result,plan))
    diagnostics.update(run_graph_visualization_diagnostic(dataset,feature_context,experiment_result,plan))
    return diagnostics


## 9 — Result assembler → `StudyResult`

Esta camada transforma o contrato experimental em um objeto pronto para apresentação. Aqui ficam EDA, agregações, heurística da condição de referência, Pareto e especificações de gráficos.

A escolha da referência é parte da **análise**, não da renderização.


In [ ]:
def build_dataset_profile(dataset):
    y=np.asarray(dataset.targets,dtype=np.float32); classes=list(dataset.classes); n,c=y.shape
    support=y.sum(0).astype(int); prevalence=support/max(n,1); max_support=max(int(support.max()),1)
    class_balance=pd.DataFrame({
        'class_name':classes,'support':support,'prevalence':prevalence,
        'imbalance_ratio_vs_max':max_support/np.maximum(support,1),
        'neg_pos_ratio':(n-support)/np.maximum(support,1),
    }).sort_values('support',ascending=False).reset_index(drop=True)
    cardinality=y.sum(1).astype(int)
    card_dist=(pd.Series(cardinality,name='labels_per_item').value_counts().sort_index().rename_axis('labels_per_item').reset_index(name='n_items'))
    labelsets=pd.Series([tuple(np.flatnonzero(row)) for row in y])
    cooc=y.T@y; union=support[:,None]+support[None,:]-cooc
    jaccard=np.divide(cooc,union,out=np.zeros_like(cooc,dtype=float),where=union>0)
    cooc_count=pd.DataFrame(cooc.astype(int),index=classes,columns=classes)
    cooc_jaccard=pd.DataFrame(jaccard,index=classes,columns=classes)
    fold_rows=[]
    for f in sorted(np.unique(dataset.fold_id)):
        s=y[dataset.fold_id==f].sum(0).astype(int)
        fold_rows.extend({'fold':int(f)+1,'class_name':cl,'support':int(v)} for cl,v in zip(classes,s))
    fold_support=pd.DataFrame(fold_rows)
    fold_matrix=fold_support.pivot(index='class_name',columns='fold',values='support').reindex(classes)
    overview=pd.DataFrame([{
        'n_items':n,'n_classes':c,'mean_label_cardinality':float(cardinality.mean()),
        'multilabel_density':float(cardinality.mean()/c),'unique_labelsets':int(labelsets.nunique()),
        'support_min':int(support.min()),'support_max':int(support.max()),
        'max_min_support_ratio':float(support.max()/max(int(support.min()),1)),
    }])
    return {
        'dataset_overview':overview,'dataset_class_balance':class_balance,
        'dataset_label_cardinality':card_dist,'dataset_cooccurrence_count':cooc_count,
        'dataset_cooccurrence_jaccard':cooc_jaccard,'dataset_fold_support':fold_support,
        'dataset_fold_support_matrix':fold_matrix,
    }


def summarize_experiment(experiment_result):
    cond=experiment_result.conditions
    obs=experiment_result.metric_observations
    by_fold=(obs.groupby(['condition_id','metric','fold'],as_index=False).value.mean())
    summary_long=by_fold.groupby(['condition_id','metric'],as_index=False).agg(mean=('value','mean'),sd=('value','std'))
    mean_w=summary_long.pivot(index='condition_id',columns='metric',values='mean').add_suffix('_mean')
    sd_w=summary_long.pivot(index='condition_id',columns='metric',values='sd').add_suffix('_sd')
    summary=cond.merge(mean_w.reset_index(),on='condition_id').merge(sd_w.reset_index(),on='condition_id')

    timing=experiment_result.timing_observations
    timing_fold=timing.groupby(['condition_id','component','fold'],as_index=False).seconds.mean()
    timing_long=timing_fold.groupby(['condition_id','component'],as_index=False).agg(mean_seconds=('seconds','mean'),sd_seconds=('seconds','std'))
    timing_w=timing_long.pivot(index='condition_id',columns='component',values='mean_seconds').add_suffix('_mean').reset_index()
    summary=summary.merge(timing_w,on='condition_id',how='left',validate='one_to_one')
    return summary_long,by_fold,summary,timing_long,timing_fold


def build_model_overview(condition_summary, primary_metrics):
    # Agregação por modelo só é comparável para modelos avaliados na mesma matriz.
    base=condition_summary.copy()
    if 'execution_scope' in base:
        base=base[base.execution_scope=='main_matrix'].copy()
    parts=[]
    for metric in primary_metrics:
        col=f'{metric}_mean'
        if col not in base:
            continue
        x=base[['representation_id','model_id','model_name',col]].copy()
        x['rank']=x.groupby('representation_id')[col].rank(ascending=False,method='min')
        x['win']=(x[col]==x.groupby('representation_id')[col].transform('max')).astype(int)
        p=x.groupby(['model_id','model_name'],as_index=False).agg(median=(col,'median'),mean_rank=('rank','mean'),wins=('win','sum'))
        p['metric']=metric; parts.append(p)
    return pd.concat(parts,ignore_index=True) if parts else pd.DataFrame()


def select_reference(condition_summary, experiment_result):
    policy=experiment_result.metadata['reference_policy']; metrics=list(policy['metrics'])
    x=condition_summary.copy()
    vals=np.column_stack([x[f'{m}_mean'].clip(lower=0).to_numpy(float) for m in metrics])
    denom=np.sum(np.divide(1.0,vals,out=np.full_like(vals,np.inf),where=vals>0),axis=1)
    hmean=np.divide(len(metrics),denom,out=np.zeros(len(x),dtype=float),where=np.isfinite(denom)&(denom>0))
    sd_sum=np.zeros(len(x),dtype=float)
    for m in metrics:
        if f'{m}_sd' in x: sd_sum+=x[f'{m}_sd'].fillna(0).to_numpy(float)
    x['reference_score']=hmean-float(policy.get('stability_weight',0))*sd_sum
    time_col=f"{policy.get('time_tiebreak_component','pipeline_incremental')}_mean"
    if time_col not in x: x[time_col]=np.nan
    x['_time_sort']=x[time_col].fillna(np.inf)
    sort_cols=['reference_score']+[f'{m}_mean' for m in reversed(metrics)]+['_time_sort']
    asc=[False]*(1+len(metrics))+[True]
    ranking=x.sort_values(sort_cols,ascending=asc).drop(columns='_time_sort').reset_index(drop=True)
    return ranking,ranking.iloc[0].to_dict()


def build_reference_by_class(experiment_result,reference,dataset_tables):
    q=experiment_result.class_metric_observations
    q=q[q.condition_id==reference['condition_id']]
    fold=(q.groupby(['class_name','metric','fold'],as_index=False).value.mean())
    s=fold.groupby(['class_name','metric'],as_index=False).agg(mean=('value','mean'),sd=('value','std'))
    mean_w=s.pivot(index='class_name',columns='metric',values='mean').add_suffix('_mean')
    sd_w=s.pivot(index='class_name',columns='metric',values='sd').add_suffix('_sd')
    out=mean_w.join(sd_w).reset_index()
    balance=dataset_tables['dataset_class_balance'][['class_name','support','prevalence','imbalance_ratio_vs_max']]
    return balance.merge(out,on='class_name',how='left',validate='one_to_one')


def pareto_front(df,score_col,time_col):
    x=df.dropna(subset=[score_col,time_col]).reset_index(drop=True)
    keep=[]; vals=x[[score_col,time_col]].to_numpy(float)
    for i,(score_i,time_i) in enumerate(vals):
        dominated=any(j!=i and sj>=score_i and tj<=time_i and (sj>score_i or tj<time_i) for j,(sj,tj) in enumerate(vals))
        if not dominated: keep.append(i)
    return x.iloc[keep].sort_values(time_col).reset_index(drop=True)


def build_plot_specs(tables,reference,experiment_result,diagnostics):
    labels=experiment_result.metadata['metric_labels']; primary=list(experiment_result.metadata['primary_metrics'])
    p0,p1=primary[:2]
    specs=[
        {'id':'dataset_support','kind':'barh','table':'dataset_class_balance','x':'support','y':'class_name','title':'Suporte por macrogênero — desenvolvimento','xlabel':'Número de obras','ylabel':'Macrogênero'},
        {'id':'dataset_cardinality','kind':'bar','table':'dataset_label_cardinality','x':'labels_per_item','y':'n_items','title':'Cardinalidade multirrótulo','xlabel':'Rótulos por obra','ylabel':'Número de obras'},
        {'id':'dataset_jaccard','kind':'heatmap','table':'dataset_cooccurrence_jaccard','title':'Coocorrência entre macrogêneros — Jaccard','colorbar':'Jaccard','vmin':0,'vmax':1},
        {'id':'dataset_fold_support','kind':'heatmap','table':'dataset_fold_support_matrix','title':'Suporte por macrogênero e fold','colorbar':'Support','annotate':True,'fmt':'.0f'},
        {'id':'top_conditions','kind':'grouped_barh','table':'reference_ranking_top','y':'condition_label','series':[f'{p0}_mean',f'{p1}_mean'],'series_labels':[labels.get(p0,p0),labels.get(p1,p1)],'title':'Condições mais fortes pela heurística de referência','xlabel':'F1','xlim':[0,1]},
        {'id':'primary_scatter','kind':'scatter','table':'condition_summary','x':f'{p0}_mean','y':f'{p1}_mean','group':'model_name','label':'representation_name','highlight':'is_reference','label_filters':['is_reference'],'title':f"{labels.get(p0,p0)} × {labels.get(p1,p1)} por condição",'xlabel':labels.get(p0,p0),'ylabel':labels.get(p1,p1),'auto_xlim':True,'auto_ylim':True,'x_bounds':[0,1],'y_bounds':[0,1]},
        {'id':'model_overview','kind':'grouped_barh','table':'model_overview_wide','y':'model_name','series':[f'{p0}_median',f'{p1}_median'],'series_labels':[labels.get(p0,p0),labels.get(p1,p1)],'title':'Visão agregada por modelo','xlabel':'F1 mediano','xlim':[0,1]},
        {'id':'reference_by_class','kind':'grouped_barh','table':'reference_by_class','y':'class_name','series':['precision_mean','recall_mean','f1_mean'],'series_labels':['Precisão','Recall','F1'],'errors':[None,None,'f1_sd'],'title':f"Desempenho por macrogênero — referência: {reference['representation_name']} × {reference['model_name']}",'xlabel':'Métrica média (barra de erro no F1 = DP entre folds)','xlim':[0,1]},
        {'id':'support_vs_reference_f1','kind':'scatter','table':'reference_by_class','x':'support','y':'f1_mean','label':'class_name','title':'Suporte × F1 por macrogênero','xlabel':'Support no desenvolvimento','ylabel':'F1 médio','trendline':True,'correlation':True,'auto_ylim':True,'y_bounds':[0,1]},
        {'id':'score_vs_time','kind':'scatter','table':'score_time','x':'pipeline_incremental_mean','y':f'{p0}_mean','group':'model_name','label':'representation_name','highlight':'is_reference','label_filters':['is_reference'],'line_filter':'is_pareto_primary','xscale':'log','title':'Desempenho × custo computacional','xlabel':'Tempo incremental médio por fold (s, escala log)','ylabel':labels.get(p0,p0),'auto_ylim':True,'y_bounds':[0,1]},
    ]

    if 'graph_operator_comparison' in diagnostics and not diagnostics['graph_operator_comparison'].empty:
        specs.append({
            'id':'graph_operator_comparison','kind':'grouped_barh','table':'diagnostic__graph_operator_comparison',
            'y':'condition_label','series':[f'{p0}_mean',f'{p1}_mean'],'series_labels':[labels.get(p0,p0),labels.get(p1,p1)],
            'errors':[f'{p0}_sd',f'{p1}_sd'],'title':'Operadores homogêneos — MiniLM + word_1 no mesmo kNN',
            'xlabel':'F1 médio (barra de erro = DP entre folds)','xlim':[0,1],
        })

    if 'hetero_structural_comparison' in diagnostics and not diagnostics['hetero_structural_comparison'].empty:
        specs.append({
            'id':'hetero_structural_comparison','kind':'grouped_barh','table':'diagnostic__hetero_structural_comparison',
            'y':'condition_label','series':[f'{p0}_mean',f'{p1}_mean'],'series_labels':[labels.get(p0,p0),labels.get(p1,p1)],
            'errors':[f'{p0}_sd',f'{p1}_sd'],'title':'Heterogeneidade — densidade lexical e atenção termo→obra',
            'xlabel':'F1 médio (barra de erro = DP entre folds)','xlim':[0,1],
        })

    if 'learning_curve__micro' in diagnostics:
        specs.append({
            'id':'learning_curve_micro','kind':'line_error','table':'diagnostic__learning_curve__micro',
            'x':'fraction_percent','y':'mean','error':'sd','group':'model_name',
            'title':'Learning curve — Micro-F1','xlabel':'Percentual do treino externo','ylabel':labels.get(p0,p0),
            'auto_ylim':True,'y_bounds':[0,1],'xtick_percent':True,
        })
    if 'learning_curve__macro' in diagnostics:
        specs.append({
            'id':'learning_curve_macro','kind':'line_error','table':'diagnostic__learning_curve__macro',
            'x':'fraction_percent','y':'mean','error':'sd','group':'model_name',
            'title':'Learning curve — Macro-F1','xlabel':'Percentual do treino externo','ylabel':labels.get(p1,p1),
            'auto_ylim':True,'y_bounds':[0,1],'xtick_percent':True,
        })
    if 'learning_curve__class_heatmap' in diagnostics:
        gain_view=diagnostics.get('learning_curve__class_gain',pd.DataFrame())
        focus_name=str(gain_view.model_name.iloc[0]) if not gain_view.empty and 'model_name' in gain_view else 'modelo relacional'
        specs.append({
            'id':'learning_curve_by_class','kind':'heatmap','table':'diagnostic__learning_curve__class_heatmap',
            'title':f'Learning curve por macrogênero — {focus_name}','colorbar':'F1','vmin':0,'vmax':1,
            'annotate':True,'fmt':'.2f','xlabel':'Percentual do treino externo','ylabel':'Macrogênero',
        })
    if 'learning_curve__class_gain' in diagnostics:
        specs.append({
            'id':'learning_curve_class_gain','kind':'barh','table':'diagnostic__learning_curve__class_gain',
            'x':'delta_f1','y':'class_name','title':'Ganho de F1 por macrogênero — 20% → 100% do treino',
            'xlabel':'Δ F1','ylabel':'Macrogênero','vline':0,
        })

    if 'graph_samples__samples' in diagnostics and 'graph_samples__nodes' in diagnostics and 'graph_samples__edges' in diagnostics:
        sample_meta=diagnostics['graph_samples__samples']
        for row in sample_meta.itertuples(index=False):
            sample_id=str(row.sample_id)
            if str(row.color_mode)=='focus_class':
                specs.append({
                    'id':sample_id,'kind':'network','table':'diagnostic__graph_samples__nodes',
                    'edges_table':'diagnostic__graph_samples__edges','filters':{'sample_id':sample_id},
                    'group':'focus_group','label':'display_name','label_flag':'show_label','center':'is_center',
                    'edge_weight':'cosine','title':str(row.title),
                    'subtitle':f'Fold {int(row.fold)} | {int(row.n_nodes)} nós | {int(row.n_edges)} arestas',
                })
            else:
                specs.append({
                    'id':sample_id,'kind':'network','table':'diagnostic__graph_samples__nodes',
                    'edges_table':'diagnostic__graph_samples__edges','filters':{'sample_id':sample_id},
                    'color_numeric':'label_jaccard_to_center','colorbar':'Jaccard de rótulos com o centro',
                    'label':'display_name','label_flag':'show_label','center':'is_center','edge_weight':'cosine',
                    'title':str(row.title),'subtitle':f'Fold {int(row.fold)} | {int(row.n_nodes)} nós | {int(row.n_edges)} arestas',
                })

    for diag_id,df in diagnostics.items():
        if diag_id.endswith('__edges') or diag_id.startswith('graph_samples__') or df.empty or not {'mean_jaccard','delta','representation_name'}.issubset(df.columns):
            continue
        specs.append({
            'id':diag_id,'kind':'scatter','table':f'diagnostic__{diag_id}',
            'x':'mean_jaccard','y':'delta','label':'representation_name',
            'title':str(df.comparison_label.iloc[0])+' — homofilia × ganho',
            'xlabel':'Jaccard médio entre vizinhos','ylabel':'Δ da métrica relacional','hline':0,
            'trendline':True,'correlation':True,'auto_xlim':True,'auto_ylim':True,
        })
    return specs


def assemble_study_result(dataset,experiment_result,diagnostics=None):
    diagnostics=diagnostics or {}
    tables=build_dataset_profile(dataset)
    summary_long,by_fold,condition_summary,timing_long,timing_by_fold=summarize_experiment(experiment_result)
    ranking,reference=select_reference(condition_summary,experiment_result)
    reference_by_class=build_reference_by_class(experiment_result,reference,tables)
    model_overview=build_model_overview(condition_summary,experiment_result.metadata['primary_metrics'])
    model_overview_wide=model_overview.pivot(index=['model_id','model_name'],columns='metric',values='median').add_suffix('_median').reset_index() if not model_overview.empty else pd.DataFrame()

    primary=experiment_result.metadata['primary_metrics'][0]
    score_time=condition_summary.copy()
    score_time['is_reference']=score_time.condition_id==reference['condition_id']
    if 'pipeline_incremental_mean' in score_time:
        pareto_primary=pareto_front(score_time,f'{primary}_mean','pipeline_incremental_mean')
        pareto_ids=set(pareto_primary.condition_id)
        score_time['is_pareto_primary']=score_time.condition_id.isin(pareto_ids)
    else:
        pareto_primary=pd.DataFrame(); score_time['is_pareto_primary']=False
    condition_summary['is_reference']=condition_summary.condition_id==reference['condition_id']
    ranking['condition_label']=ranking.representation_name.astype(str)+' × '+ranking.model_name.astype(str)
    ranking_top=ranking.head(15).sort_values('reference_score').reset_index(drop=True)

    tables.update({
        'conditions':experiment_result.conditions,
        'metric_summary_long':summary_long,'metric_by_fold':by_fold,
        'condition_summary':condition_summary,'timing_summary_long':timing_long,'timing_by_fold':timing_by_fold,
        'model_overview':model_overview,'model_overview_wide':model_overview_wide,
        'reference_ranking':ranking,'reference_ranking_top':ranking_top,
        'reference_by_class':reference_by_class,'score_time':score_time,'pareto_primary':pareto_primary,
    })
    for k,v in diagnostics.items():
        tables[f'diagnostic__{k}']=v

    table_specs=[
        {'table':'dataset_overview','title':'Dados de desenvolvimento'},
        {'table':'dataset_class_balance','title':'Balanceamento por macrogênero'},
        {'table':'reference_ranking','title':'Top condições pela heurística','limit':15},
        {'table':'reference_by_class','title':'Métricas por macrogênero da condição de referência'},
        {'table':'model_overview','title':'Visão agregada dos modelos'},
        {'table':'pareto_primary','title':'Fronteira de Pareto da métrica primária'},
    ]
    if 'graph_operator_comparison' in diagnostics and not diagnostics['graph_operator_comparison'].empty:
        specs.append({
            'id':'graph_operator_comparison','kind':'grouped_barh','table':'diagnostic__graph_operator_comparison',
            'y':'condition_label','series':[f'{p0}_mean',f'{p1}_mean'],'series_labels':[labels.get(p0,p0),labels.get(p1,p1)],
            'errors':[f'{p0}_sd',f'{p1}_sd'],'title':'Operadores homogêneos — MiniLM + word_1 no mesmo kNN',
            'xlabel':'F1 médio (barra de erro = DP entre folds)','xlim':[0,1],
        })

    if 'hetero_structural_comparison' in diagnostics and not diagnostics['hetero_structural_comparison'].empty:
        table_specs.append({'table':'diagnostic__hetero_structural_comparison','title':'Experimento heterogêneo — comparação focal'})
    if 'hetero_term_structure' in diagnostics and not diagnostics['hetero_term_structure'].empty:
        table_specs.append({'table':'diagnostic__hetero_term_structure','title':'Grafo heterogêneo — estrutura por fold'})
    if 'learning_curve' in diagnostics:
        table_specs.append({'table':'diagnostic__learning_curve','title':'Learning curve — resumo global'})
    if 'learning_curve__class_gain' in diagnostics:
        table_specs.append({'table':'diagnostic__learning_curve__class_gain','title':'Learning curve — ganho por macrogênero'})
    if 'graph_samples__samples' in diagnostics:
        table_specs.append({'table':'diagnostic__graph_samples__samples','title':'Amostras qualitativas dos grafos'})

    plots=build_plot_specs(tables,reference,experiment_result,diagnostics)
    metadata={
        'study_version':STUDY_VERSION,'experiment_schema':experiment_result.schema_version,
        'primary_metrics':experiment_result.metadata['primary_metrics'],
        'metric_labels':experiment_result.metadata['metric_labels'],
        'reference_policy':experiment_result.metadata['reference_policy'],
        'dataset':dataset.metadata,'audit_used':False,
        'analysis_version':'agnostic_renderer_v4',
        'diagnostics':sorted(diagnostics.keys()),
        'learning_curve':LEARNING_CURVE_CONFIG if 'learning_curve' in diagnostics else None,
        'graph_visualization':GRAPH_VIS_CONFIG if 'graph_samples__samples' in diagnostics else None,
    }
    return StudyResult(RESULT_SCHEMA_VERSION,metadata,tables,reference,table_specs,plots)


## 10 — Apresentação agnóstica → `render_study(result)`

O renderer continua sem conhecer MiniLM, GNN, learning curve ou qualquer experimento específico. Ele interpreta somente `table_specs` e `plot_specs` do `StudyResult`.

A versão visual atual adiciona recursos genéricos reutilizáveis: limites automáticos, escala log, barras de erro, linhas com erro, tendência linear/correlação e heatmaps anotados. Isso melhora os gráficos atuais sem aumentar o acoplamento entre apresentação e experimentação.


In [ ]:
import matplotlib.pyplot as plt


def validate_study_result(result):
    if not isinstance(result,StudyResult):
        raise TypeError('render_study espera StudyResult.')
    if not isinstance(result.tables,dict) or not isinstance(result.plot_specs,list):
        raise ValueError('StudyResult inválido.')
    for spec in result.table_specs:
        if spec['table'] not in result.tables:
            raise KeyError(f"Tabela ausente: {spec['table']}")
    for spec in result.plot_specs:
        if spec['table'] not in result.tables:
            raise KeyError(f"Tabela de gráfico ausente: {spec['table']}")
        if spec.get('edges_table') and spec['edges_table'] not in result.tables:
            raise KeyError(f"Tabela de arestas ausente: {spec['edges_table']}")
    return True


def _save_rendered_figure(fig,out_dir,plot_id,dpi=160):
    out_dir=Path(out_dir); out_dir.mkdir(parents=True,exist_ok=True)
    path=out_dir/f'{plot_id}.png'; fig.tight_layout(); fig.savefig(path,dpi=dpi,bbox_inches='tight')
    plt.show(); plt.close(fig); return path


def _auto_limits(values,bounds=None,pad=0.08,log=False):
    arr=np.asarray(values,dtype=float); arr=arr[np.isfinite(arr)]
    if log: arr=arr[arr>0]
    if len(arr)==0: return None
    lo=float(arr.min()); hi=float(arr.max())
    if log:
        if lo==hi: lo*=0.8; hi*=1.2
        else: lo/=1.15; hi*=1.15
    else:
        span=hi-lo
        if span<1e-12: span=max(abs(hi)*0.1,0.05)
        lo-=pad*span; hi+=pad*span
    if bounds is not None:
        lo=max(float(bounds[0]),lo); hi=min(float(bounds[1]),hi)
    if hi<=lo: hi=lo+1e-6
    return lo,hi


def _label_subset(df,spec):
    filters=spec.get('label_filters') or []
    if filters:
        mask=np.zeros(len(df),dtype=bool)
        for col in filters:
            if col in df: mask |= df[col].fillna(False).astype(bool).to_numpy()
        return df.loc[mask]
    highlight=spec.get('highlight')
    if highlight and highlight in df and df[highlight].fillna(False).any():
        return df[df[highlight].fillna(False)]
    if len(df)>20:
        return pd.concat([df.nlargest(4,spec['y']),df.nsmallest(4,spec['y'])]).drop_duplicates()
    return df


def _add_trend_and_correlation(ax,df,spec):
    if not (spec.get('trendline') or spec.get('correlation')):
        return
    x=pd.to_numeric(df[spec['x']],errors='coerce').to_numpy(float)
    y=pd.to_numeric(df[spec['y']],errors='coerce').to_numpy(float)
    mask=np.isfinite(x)&np.isfinite(y)
    x=x[mask]; y=y[mask]
    if len(x)<2 or np.std(x)<1e-12 or np.std(y)<1e-12:
        return
    if spec.get('trendline'):
        coef=np.polyfit(x,y,1); grid=np.linspace(x.min(),x.max(),100)
        ax.plot(grid,coef[0]*grid+coef[1],linestyle='--',linewidth=1.3,label='Tendência linear')
    if spec.get('correlation'):
        r=float(np.corrcoef(x,y)[0,1])
        ax.text(.02,.98,f'Pearson r = {r:.2f}',transform=ax.transAxes,va='top',ha='left',fontsize=9,
                bbox={'boxstyle':'round','facecolor':'white','alpha':.75,'edgecolor':'none'})


def _filter_table(df, filters):
    out=df
    for col,val in (filters or {}).items():
        if col in out.columns: out=out[out[col].astype(str)==str(val)]
    return out.copy()


def _render_network(spec,tables,out_dir):
    nodes=_filter_table(tables[spec['table']],spec.get('filters'))
    edges=_filter_table(tables[spec['edges_table']],spec.get('filters')) if spec.get('edges_table') else pd.DataFrame()
    if nodes.empty:
        raise ValueError(f"Amostra de grafo vazia: {spec['id']}")
    fig,ax=plt.subplots(figsize=(9,7))
    pos={int(r.node_index):(float(r.x),float(r.y)) for r in nodes.itertuples(index=False)}

    if not edges.empty:
        wcol=spec.get('edge_weight')
        vals=pd.to_numeric(edges[wcol],errors='coerce').fillna(0).to_numpy(float) if wcol and wcol in edges else np.ones(len(edges))
        if len(vals):
            lo,hi=float(np.nanmin(vals)),float(np.nanmax(vals)); span=max(hi-lo,1e-9)
        else: lo=0.0; span=1.0
        for r,w in zip(edges.itertuples(index=False),vals):
            s=int(r.source_index); t=int(r.target_index)
            if s not in pos or t not in pos: continue
            x0,y0=pos[s]; x1,y1=pos[t]
            width=.45+1.8*max(0.0,min(1.0,(float(w)-lo)/span))
            ax.plot([x0,x1],[y0,y1],linewidth=width,alpha=.28,color='0.55',zorder=1)

    color_numeric=spec.get('color_numeric')
    group=spec.get('group')
    if color_numeric and color_numeric in nodes:
        from matplotlib.colors import Normalize
        vals=pd.to_numeric(nodes[color_numeric],errors='coerce').fillna(0).to_numpy(float)
        norm=Normalize(vmin=float(np.nanmin(vals)),vmax=float(np.nanmax(vals)) if np.nanmax(vals)>np.nanmin(vals) else float(np.nanmin(vals))+1e-6)
        sc=ax.scatter(nodes.x,nodes.y,c=vals,cmap='viridis',norm=norm,s=90,edgecolors='white',linewidths=.8,zorder=3)
        fig.colorbar(sc,ax=ax,label=spec.get('colorbar',color_numeric))
    elif group and group in nodes:
        for name,g in nodes.groupby(group,sort=True):
            ax.scatter(g.x,g.y,s=90,edgecolors='white',linewidths=.8,label=str(name),zorder=3)
    else:
        ax.scatter(nodes.x,nodes.y,s=90,edgecolors='white',linewidths=.8,zorder=3)

    center_col=spec.get('center')
    if center_col and center_col in nodes and nodes[center_col].fillna(False).any():
        c=nodes[nodes[center_col].fillna(False)]
        ax.scatter(c.x,c.y,marker='*',s=320,facecolors='none',edgecolors='black',linewidths=1.7,label='Centro',zorder=5)

    label_col=spec.get('label'); flag=spec.get('label_flag')
    if label_col and label_col in nodes:
        lab=nodes[nodes[flag].fillna(False)] if flag and flag in nodes else nodes.head(12)
        for r in lab.itertuples(index=False):
            txt=str(getattr(r,label_col)); txt=txt if len(txt)<=28 else txt[:25]+'…'
            ax.annotate(txt,(float(r.x),float(r.y)),xytext=(5,5),textcoords='offset points',fontsize=7,zorder=6)

    title=spec.get('title','')
    subtitle=spec.get('subtitle')
    ax.set_title(title+(f'\n{subtitle}' if subtitle else ''))
    ax.set_xticks([]); ax.set_yticks([]); ax.set_xlabel(''); ax.set_ylabel('')
    ax.set_aspect('equal',adjustable='datalim')
    for spine in ax.spines.values(): spine.set_visible(False)
    handles,labels=ax.get_legend_handles_labels()
    if handles: ax.legend(fontsize=8,loc='best')
    return _save_rendered_figure(fig,out_dir,spec['id'])


def _render_plot_spec(spec,tables,out_dir):
    df=_filter_table(tables[spec['table']],spec.get('filters'))
    kind=spec['kind']; fig=None
    if kind=='network':
        return _render_network(spec,tables,out_dir)
    if kind in ('bar','barh'):
        fig,ax=plt.subplots(figsize=(9,6))
        if kind=='barh': ax.barh(df[spec['y']],df[spec['x']])
        else: ax.bar(df[spec['x']].astype(str),df[spec['y']])
    elif kind=='grouped_barh':
        fig,ax=plt.subplots(figsize=(10,max(5,0.36*len(df)+2)))
        y=np.arange(len(df)); series=spec['series']; labels=spec.get('series_labels',series); h=.75/max(len(series),1)
        errors=spec.get('errors',[None]*len(series)); offsets=(np.arange(len(series))-(len(series)-1)/2)*h
        for off,col,label,err_col in zip(offsets,series,labels,errors):
            xerr=df[err_col] if err_col and err_col in df else None
            ax.barh(y+off,df[col],height=h,label=label,xerr=xerr,capsize=2 if xerr is not None else 0)
        ax.set_yticks(y,df[spec['y']])
    elif kind=='scatter':
        fig,ax=plt.subplots(figsize=(8,6))
        group=spec.get('group')
        if group and group in df:
            for name,g in df.groupby(group): ax.scatter(g[spec['x']],g[spec['y']],label=str(name),alpha=.65)
        else: ax.scatter(df[spec['x']],df[spec['y']],alpha=.7)
        label_col=spec.get('label')
        if label_col and label_col in df:
            label_df=_label_subset(df,spec)
            for row in label_df.itertuples(index=False):
                ax.annotate(str(getattr(row,label_col)),(getattr(row,spec['x']),getattr(row,spec['y'])),xytext=(4,4),textcoords='offset points',fontsize=7)
        highlight=spec.get('highlight')
        if highlight and highlight in df and df[highlight].fillna(False).any():
            hdf=df[df[highlight].fillna(False)]
            ax.scatter(hdf[spec['x']],hdf[spec['y']],marker='*',s=220,label='Referência')
        line_filter=spec.get('line_filter')
        if line_filter and line_filter in df and df[line_filter].fillna(False).any():
            ldf=df[df[line_filter].fillna(False)].sort_values(spec['x'])
            ax.plot(ldf[spec['x']],ldf[spec['y']],marker='o',label='Pareto')
        _add_trend_and_correlation(ax,df,spec)
    elif kind=='line_error':
        fig,ax=plt.subplots(figsize=(8,6))
        group=spec.get('group')
        groups=df.groupby(group) if group and group in df else [('Série',df)]
        for name,g in groups:
            g=g.sort_values(spec['x']); err=g[spec['error']] if spec.get('error') in g else None
            ax.errorbar(g[spec['x']],g[spec['y']],yerr=err,marker='o',capsize=3,label=str(name))
        if spec.get('xtick_percent'):
            ticks=sorted(pd.unique(df[spec['x']].dropna()))
            ax.set_xticks(ticks,[f'{int(t)}%' for t in ticks])
    elif kind=='heatmap':
        fig,ax=plt.subplots(figsize=(9,8)); arr=df.to_numpy(float)
        im=ax.imshow(arr,aspect='auto',vmin=spec.get('vmin'),vmax=spec.get('vmax'))
        ax.set_xticks(range(len(df.columns)),[str(x) for x in df.columns],rotation=90)
        ax.set_yticks(range(len(df.index)),[str(x) for x in df.index])
        if spec.get('annotate'):
            fmt=spec.get('fmt','.0f')
            for i in range(arr.shape[0]):
                for j in range(arr.shape[1]):
                    if np.isfinite(arr[i,j]): ax.text(j,i,format(arr[i,j],fmt),ha='center',va='center',fontsize=7)
        fig.colorbar(im,ax=ax,label=spec.get('colorbar',''))
    else:
        raise ValueError(f'Tipo de gráfico não suportado: {kind}')

    if spec.get('xscale'): ax.set_xscale(spec['xscale'])
    if spec.get('yscale'): ax.set_yscale(spec['yscale'])
    ax.set_title(spec.get('title',''))
    if spec.get('xlabel'): ax.set_xlabel(spec['xlabel'])
    if spec.get('ylabel'): ax.set_ylabel(spec['ylabel'])
    if spec.get('xlim'): ax.set_xlim(*spec['xlim'])
    elif spec.get('auto_xlim') and spec.get('x') in df:
        lim=_auto_limits(df[spec['x']],spec.get('x_bounds'),log=spec.get('xscale')=='log')
        if lim: ax.set_xlim(*lim)
    if spec.get('ylim'): ax.set_ylim(*spec['ylim'])
    elif spec.get('auto_ylim') and spec.get('y') in df:
        lim=_auto_limits(df[spec['y']],spec.get('y_bounds'),log=spec.get('yscale')=='log')
        if lim: ax.set_ylim(*lim)
    if 'hline' in spec: ax.axhline(spec['hline'],linewidth=1)
    if 'vline' in spec: ax.axvline(spec['vline'],linewidth=1)
    handles,labels=ax.get_legend_handles_labels()
    if handles: ax.legend(fontsize=8)
    return _save_rendered_figure(fig,out_dir,spec['id'])


def persist_study_result(result,directory):
    validate_study_result(result)
    directory=Path(directory); table_dir=directory/'tables'; table_dir.mkdir(parents=True,exist_ok=True)
    for name,df in result.tables.items():
        if isinstance(df,pd.DataFrame): df.to_csv(table_dir/f'{name}.csv',index=True if df.index.name or not isinstance(df.index,pd.RangeIndex) else False)
    meta={**result.metadata,'schema_version':result.schema_version,'reference':result.reference,'tables':sorted(result.tables),'plots':[p['id'] for p in result.plot_specs]}
    (directory/'study_result.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False,default=str),encoding='utf-8')


def render_study(result,output_dir):
    validate_study_result(result)
    print('=== STUDY RESULT ===')
    print('Schema:',result.schema_version)
    print('Referência exploratória:',result.reference.get('representation_name'),'×',result.reference.get('model_name'))
    for spec in result.table_specs:
        print(f"\n=== {spec['title']} ===")
        df=result.tables[spec['table']]
        display(df.head(spec['limit']) if spec.get('limit') else df)
    fig_dir=Path(output_dir)/'figures'; paths=[]
    for spec in result.plot_specs:
        paths.append(_render_plot_spec(spec,result.tables,fig_dir))
    print(f'\n{len(paths)} figuras salvas em: {fig_dir}')
    return paths


## 11 — Pipeline completa

Fluxo único: preparar representações → executar matriz principal + experimento heterogêneo → produzir diagnósticos (incluindo learning curve) → montar `StudyResult` → renderizar. Tanto a matriz principal quanto a learning curve têm checkpoints e podem ser retomadas após interrupção.


In [ ]:
print('1/5 — preparando representações')
FEATURE_CONTEXT=prepare_feature_context(DATASET)
display(FEATURE_CONTEXT.catalog)

print('2/5 — sanity checks')
assert run_sanity_checks(DATASET,FEATURE_CONTEXT,EXPERIMENT_PLAN)
print('Sanity checks: OK')

print('3/5 — executando matriz principal + baselines homogêneos focais + ablações heterogêneas')
EXPERIMENT_RESULT=execute_experiments(DATASET,FEATURE_CONTEXT,EXPERIMENT_PLAN)
print(f'ExperimentResult: {len(EXPERIMENT_RESULT.conditions)} condições | {len(EXPERIMENT_RESULT.metric_observations)} observações de métricas')

print('4/5 — diagnósticos opcionais (relacional + learning curve + amostras de grafo) e montagem do StudyResult')
DIAGNOSTICS=run_diagnostics(DATASET,FEATURE_CONTEXT,EXPERIMENT_RESULT,EXPERIMENT_PLAN)
STUDY_RESULT=assemble_study_result(DATASET,EXPERIMENT_RESULT,DIAGNOSTICS)
REPORT_DIR=STUDY_DIR/'report'
persist_study_result(STUDY_RESULT,REPORT_DIR)

print('5/5 — apresentação')
FIGURES=render_study(STUDY_RESULT,REPORT_DIR)


## 12 — Manifesto

O manifesto registra separadamente o contrato experimental, o contrato de apresentação e o protocolo da learning curve. A learning curve é um diagnóstico exploratório do desenvolvimento; não altera a escolha da referência e não usa a auditoria.


In [ ]:
def _jsonable_cfg(cfg):
    out={}
    for k,v in cfg.items():
        if isinstance(v,tuple): out[k]=list(v)
        elif isinstance(v,(str,int,float,bool,type(None),list,dict)): out[k]=v
        else: out[k]=str(v)
    return out

manifest={
    'status':'COMPLETE_DECOUPLED_STUDY_WITH_GRAPH_OPERATOR_AND_HETERO_DIFFUSION_ABLATIONS', 'study_version':STUDY_VERSION, 'run_id':RUN_ID,
    'architecture':{
        'dataset_contract':'DatasetContext','experiment_contract':EXPERIMENT_RESULT.schema_version,
        'presentation_contract':STUDY_RESULT.schema_version,
        'renderer_entrypoint':'render_study(StudyResult)',
    },
    'dataset':DATASET.metadata,'audit_used':False,'classes':list(DATASET.classes),
    'cv':{'outer_splits':len(DATASET.folds),'group':'grupo_franquia','fold_seed':CV_SEED,'model_seeds':{m:list(s) for m,s in EXPERIMENT_PLAN.model_seeds.items()}},
    'device':'cpu','runtime':RUNTIME_INFO,
    'training':{'epochs':EPOCHS,'hidden':HIDDEN,'dropout':DROPOUT,'lr':LR,'weight_decay':WEIGHT_DECAY,'threshold':FIXED_THRESHOLD,'loss':'BCEWithLogitsLoss_pos_weight_train_only'},
    'graph':EXPERIMENT_PLAN.graph_config,'models':EXPERIMENT_PLAN.model_metadata,
    'model_execution_enabled':{m:bool(v.get('enabled',True)) for m,v in MODEL_METADATA.items()},
    'special_experiments':[dict(x) for x in EXPERIMENT_PLAN.special_experiments],
    'homogeneous_operator_config':HOMOGENEOUS_OPERATOR_CFG,
    'hetero_term_config':HETERO_TERM_CFG,
    'hetero_diffusion_config':HETERO_DIFFUSION_CFG,
    'incremental_reuse':{'enabled':INCREMENTAL_REUSE,'base_run_id':INCREMENTAL_BASE_RUN_ID},
    'relational_research_registry':RELATIONAL_RESEARCH_REGISTRY,
    'sources':list(SOURCE_ORDER),'source_components':{k:list(v) for k,v in SOURCE_COMPONENTS.items()},
    'feature_extractors':FEATURE_CONTEXT.fixed_meta,
    'n_bundles':len(EXPERIMENT_PLAN.registry),'n_conditions':len(EXPERIMENT_RESULT.conditions),
    'primary_metrics':list(EXPERIMENT_PLAN.primary_metrics),'reference_policy':EXPERIMENT_PLAN.reference_policy,
    'reference':STUDY_RESULT.reference,
    'analysis_version':'agnostic_renderer_v5',
    'diagnostics':sorted(DIAGNOSTICS.keys()),
    'learning_curve':LEARNING_CURVE_CONFIG,
    'graph_visualization':GRAPH_VIS_CONFIG,
    'target_labels_used_to_construct_features_or_edges':False,
    'labels_used_posthoc_for_diagnostics_only':True,
    'graph_visualization_labels_posthoc_only':True,
    'learning_curve_subset_selection_uses_outer_train_labels_only':True,
    'multiple_comparisons_note':'exploratory development screening; reference is descriptive, not final unbiased evidence',
    'generative_calls':0,
    'contract_files':['conditions.csv','metric_observations.csv','class_metric_observations.csv','timing_observations.csv','learning_curve_metrics.csv','learning_curve_class_metrics.csv','learning_curve_subsets.csv','learning_curve_subset_support.csv','report/study_result.json'],
}
(STUDY_DIR/'manifesto.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
(RUN_DIR/'manifesto.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
print('Estudo concluído:',RUN_ID)
print('Pasta:',STUDY_DIR)
print('Contrato final:',STUDY_RESULT.schema_version)
